<a href="https://colab.research.google.com/github/lidwaa/cinetrack/blob/main/ocr_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 OCR Benchmark Dashboard
Ce notebook charge tous vos fichiers `results_nomduparseur.csv` depuis un dossier,
agrège les données, calcule des stats et génère un dashboard HTML interactif.

## ⚙️ Configuration

In [ ]:
# ─── PARAMÈTRES À MODIFIER ───────────────────────────────────────────────
CSV_FOLDER  = "./csv_results"      # chemin vers votre dossier de CSV
OUTPUT_HTML = "./ocr_dashboard.html"  # fichier HTML généré

# ─── SPECS TECHNIQUES DU BENCHMARK ──────────────────────────────────────
# Remplissez les champs que vous voulez voir apparaître dans le footer.
# Laissez une valeur vide "" pour masquer un champ.
BENCHMARK_SPECS = {
    # ── Environnement ──────────────────────────────────────────────────
    "date":            "2026-04-24",
    "os":              "Ubuntu 22.04 LTS",
    "python_version":  "3.11.4",
    "cpu":             "Intel Xeon E5-2690 v4 × 28 cores",
    "gpu":             "NVIDIA A100 40GB",
    "ram":             "128 GB",

    # ── Pipeline ───────────────────────────────────────────────────────
    "pipeline_version": "v1.2.0",
    "dataset":          "Internal dataset — 320 documents",
    "dataset_split":    "80% train / 20% test",
    "eval_metric":      "GT Accuracy, Fill Rate, LLM Judge Score, Latency",
    "n_runs":           "3 runs averaged",

    # ── Packages Python ────────────────────────────────────────────────
    # Format : "nom_package": "version"
    "packages": {
        "pandas":         "2.2.1",
        "numpy":          "1.26.4",
        "mistralai":      "1.0.3",
        "openai":         "1.23.0",
        "paddleocr":      "2.7.3",
        "pillow":         "10.3.0",
        "pdfplumber":     "0.11.0",
        "transformers":   "4.40.1",
        "torch":          "2.3.0",
    },

    # ── Modèles OCR testés ─────────────────────────────────────────────
    "ocr_models": {
        "markildown":  "Markdown-based layout parser",
        "liteparse":   "Lightweight heuristic parser",
        "paddleocr":   "PaddleOCR v2.7 — PP-OCRv4",
        "litepaddle":  "PaddleOCR lite variant",
        "mistral_ocr": "Mistral OCR API — mistral-ocr-latest",
    },

    # ── Parseurs LLM testés ────────────────────────────────────────────
    "llm_parsers": {
        "mistral_medium": "mistral-medium-2505",
        "gemma_4":        "gemma-3-27b-it",
        "gpt4o":          "gpt-4o-2024-11-20",
    },

    # ── Notes libres ───────────────────────────────────────────────────
    "notes": "Temperature=0 for all LLM parsers. Timeout=60s per document.",
}
# ─────────────────────────────────────────────────────────────────────────

## 📦 Imports

In [ ]:
import os
import re
import json
import warnings
from pathlib import Path

import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')
print('✅ Imports OK')

## 📂 Chargement des CSV

In [ ]:
def load_benchmark_folder(folder: str) -> pd.DataFrame:
    """
    Charge tous les fichiers results_*.csv du dossier.
    Extrait le nom du parseur depuis le nom du fichier.
    Retourne un DataFrame agrégé.
    """
    folder = Path(folder)
    assert folder.exists(), f"Dossier introuvable : {folder.resolve()}"

    pattern = re.compile(r'results[_-](.+)\.csv$', re.IGNORECASE)
    dfs = []

    csv_files = sorted(folder.glob('*.csv'))
    assert len(csv_files) > 0, f"Aucun fichier CSV trouvé dans {folder.resolve()}"

    for filepath in csv_files:
        match = pattern.match(filepath.name)
        parser_name = match.group(1) if match else filepath.stem

        df = pd.read_csv(filepath)
        df.columns = df.columns.str.strip()
        df['parser'] = parser_name
        df['source_file'] = filepath.name
        dfs.append(df)
        print(f'  ✅  {filepath.name:<45} → parseur: "{parser_name}"  ({len(df)} lignes)')

    combined = pd.concat(dfs, ignore_index=True)
    print(f'\n📊 Total : {len(combined)} entrées · {combined["parser"].nunique()} parseur(s) · {combined["model"].nunique()} modèle(s) OCR')
    return combined


print(f'Chargement depuis : {Path(CSV_FOLDER).resolve()}\n')
df = load_benchmark_folder(CSV_FOLDER)
df.head()

## 🔍 Aperçu & validation des données

In [ ]:
EXPECTED_COLS = ['model', 'avg_latency_s', 'fill_rate_pct', 'llm_judge_score_pct', 'gt_accuracy_pct']

print('=== Colonnes détectées ===')
print(list(df.columns))

missing = [c for c in EXPECTED_COLS if c not in df.columns]
if missing:
    print(f'\n⚠️  Colonnes manquantes : {missing}')
    print('   Le dashboard ignorera ces métriques.')
else:
    print('\n✅ Toutes les colonnes attendues sont présentes.')

print('\n=== Valeurs manquantes ===')
print(df[EXPECTED_COLS].isnull().sum())

print('\n=== Types ===')
print(df[EXPECTED_COLS].dtypes)

## 📈 Stats résumé par parseur

In [ ]:
metrics = [c for c in ['gt_accuracy_pct', 'fill_rate_pct', 'llm_judge_score_pct', 'avg_latency_s'] if c in df.columns]

summary = df.groupby('parser')[metrics].agg(['mean', 'min', 'max']).round(2)
summary.columns = ['_'.join(c) for c in summary.columns]

print('=== Résumé par parseur ===')
display(summary)

print('\n=== Meilleur modèle OCR par parseur (GT Accuracy) ===')
if 'gt_accuracy_pct' in df.columns:
    best = df.loc[df.groupby('parser')['gt_accuracy_pct'].idxmax(), ['parser', 'model', 'gt_accuracy_pct', 'avg_latency_s']]
    display(best.reset_index(drop=True))

## 🏆 Classement global

In [ ]:
if 'gt_accuracy_pct' in df.columns:
    ranking = (
        df.groupby(['model', 'parser'])
        .agg({m: 'mean' for m in metrics})
        .round(2)
        .sort_values('gt_accuracy_pct', ascending=False)
        .reset_index()
    )
    ranking.index = ranking.index + 1
    ranking.index.name = 'rank'
    print('=== Classement global (par GT Accuracy décroissant) ===')
    display(ranking)

## 🌐 Génération du Dashboard HTML

In [ ]:
def generate_dashboard(df: pd.DataFrame, output_path: str, specs: dict = None):
    """
    Sérialise les données + specs en JSON et les injecte dans le template HTML.
    Écrit le fichier HTML final.
    """
    records = df.where(pd.notnull(df), None).to_dict(orient='records')
    data_json  = json.dumps(records, ensure_ascii=False)
    specs_json = json.dumps(specs or {}, ensure_ascii=False)

    html = HTML_TEMPLATE.replace('__DATA_JSON__', data_json).replace('__SPECS_JSON__', specs_json)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f'✅ Dashboard généré : {Path(output_path).resolve()}')
    print(f'   {len(records)} entrées · {df["parser"].nunique()} parseur(s)')


# ─── Template HTML (données injectées via __DATA_JSON__ et __SPECS_JSON__) ─
HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>OCR Benchmark Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap');
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
:root {
  --bg: #0d0f14; --surface: #141720; --surface2: #1c2030;
  --border: rgba(255,255,255,0.07); --border2: rgba(255,255,255,0.14);
  --text: #e8eaf0; --muted: #6b7080;
  --accent: #4f9cf9; --accent2: #7b6ef6;
  --success: #4ade80; --warning: #fbbf24; --danger: #f87171; --teal: #2dd4bf;
  --mono: 'Space Mono', monospace; --sans: 'DM Sans', sans-serif;
}
body { background: var(--bg); color: var(--text); font-family: var(--sans); font-size: 14px; min-height: 100vh; }
.header { border-bottom:1px solid var(--border); padding:20px 32px; display:flex; align-items:center; justify-content:space-between; background:var(--surface); position:sticky; top:0; z-index:10; }
.header h1 { font-family:var(--mono); font-size:17px; font-weight:700; }
.header p { color:var(--muted); font-size:11px; margin-top:3px; font-family:var(--mono); }
.badge { background:rgba(79,156,249,0.12); color:var(--accent); border:1px solid rgba(79,156,249,0.25); padding:4px 10px; border-radius:4px; font-family:var(--mono); font-size:11px; }
.main { max-width:1400px; margin:0 auto; padding:28px 32px; }
.filters-bar { display:flex; gap:12px; margin-bottom:24px; flex-wrap:wrap; align-items:center; background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:14px 18px; }
.filter-label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; }
.parser-toggle { padding:5px 12px; border-radius:20px; border:1px solid var(--border2); background:transparent; color:var(--muted); font-size:12px; font-family:var(--mono); cursor:pointer; transition:all .15s; }
.parser-toggle.active { color:#0d0f14; border-color:transparent; font-weight:700; }
select { background:var(--surface); border:1px solid var(--border2); color:var(--text); padding:7px 12px; border-radius:6px; font-family:var(--sans); font-size:13px; cursor:pointer; outline:none; }
select:focus { border-color:var(--accent); }
.btn { background:var(--accent); color:#0d0f14; border:none; padding:8px 16px; border-radius:6px; font-family:var(--sans); font-size:13px; font-weight:600; cursor:pointer; transition:opacity .15s; }
.btn:hover { opacity:.85; }
.btn-ghost { background:transparent; color:var(--text); border:1px solid var(--border2); }
.btn-ghost:hover { background:var(--surface2); }
.kpi-row { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:16px; margin-bottom:32px; }
.kpi-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; position:relative; overflow:hidden; }
.kpi-card::before { content:''; position:absolute; top:0; left:0; right:0; height:2px; }
.kpi-card.c-blue::before { background:var(--accent); } .kpi-card.c-green::before { background:var(--success); }
.kpi-card.c-purple::before { background:var(--accent2); } .kpi-card.c-teal::before { background:var(--teal); }
.kpi-card .label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; margin-bottom:10px; }
.kpi-card .value { font-size:28px; font-weight:600; font-family:var(--mono); line-height:1; }
.kpi-card .sub { font-size:11px; color:var(--muted); margin-top:6px; font-family:var(--mono); }
.kpi-card.c-blue .value{color:var(--accent)} .kpi-card.c-green .value{color:var(--success)} .kpi-card.c-purple .value{color:var(--accent2)} .kpi-card.c-teal .value{color:var(--teal)}
.section-title { font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; color:var(--muted); margin-bottom:16px; display:flex; align-items:center; gap:10px; }
.section-title::after { content:''; flex:1; height:1px; background:var(--border); }
.chart-grid { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:32px; }
@media(max-width:900px){.chart-grid{grid-template-columns:1fr;}}
.chart-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; }
.chart-card h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-card .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.chart-full { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; margin-bottom:32px; }
.chart-full h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-full .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.legend { display:flex; flex-wrap:wrap; gap:14px; margin-bottom:10px; }
.legend-item { display:flex; align-items:center; gap:6px; font-size:12px; font-family:var(--mono); color:var(--muted); }
.legend-dot { width:10px; height:10px; border-radius:2px; }
.table-wrap { background:var(--surface); border:1px solid var(--border); border-radius:10px; overflow:hidden; margin-bottom:32px; }
.table-header { padding:16px 20px; border-bottom:1px solid var(--border); display:flex; align-items:center; justify-content:space-between; }
.table-header h3 { font-family:var(--mono); font-size:13px; font-weight:500; }
table { width:100%; border-collapse:collapse; }
thead th { padding:10px 16px; text-align:left; font-family:var(--mono); font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.5px; background:var(--surface2); cursor:pointer; user-select:none; white-space:nowrap; }
thead th:hover { color:var(--text); }
tbody tr { border-top:1px solid var(--border); transition:background .1s; }
tbody tr:hover { background:var(--surface2); }
tbody td { padding:11px 16px; font-size:13px; white-space:nowrap; }
.td-model { font-family:var(--mono); font-weight:700; }
.td-parser { font-family:var(--mono); font-size:11px; padding:3px 8px; border-radius:4px; }
.bar-cell { display:flex; align-items:center; gap:10px; }
.bar-track { flex:1; max-width:90px; height:4px; background:var(--surface2); border-radius:2px; overflow:hidden; }
.bar-fill { height:100%; border-radius:2px; }
.td-val { font-family:var(--mono); font-size:13px; min-width:45px; }
.rank-badge { display:inline-flex; align-items:center; justify-content:center; width:22px; height:22px; border-radius:4px; font-family:var(--mono); font-size:11px; font-weight:700; }
.rank-1{background:rgba(251,191,36,.15);color:var(--warning);border:1px solid rgba(251,191,36,.3)}
.rank-2{background:rgba(148,163,184,.1);color:#94a3b8;border:1px solid rgba(148,163,184,.2)}
.rank-3{background:rgba(180,130,80,.1);color:#b48250;border:1px solid rgba(180,130,80,.2)}
.rank-other{background:var(--surface2);color:var(--muted);border:1px solid var(--border)}

/* ── FOOTER ── */
.footer { border-top:1px solid var(--border); background:var(--surface); margin-top:16px; }
.footer-toggle { width:100%; display:flex; align-items:center; justify-content:space-between; padding:16px 32px; background:transparent; border:none; color:var(--muted); font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; cursor:pointer; transition:color .15s; }
.footer-toggle:hover { color:var(--text); }
.footer-toggle .arrow { transition:transform .25s; font-size:14px; }
.footer-toggle.open .arrow { transform:rotate(180deg); }
.footer-body { display:none; padding:0 32px 32px; max-width:1400px; margin:0 auto; }
.footer-body.open { display:block; }
.specs-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:20px; margin-top:8px; }
.spec-card { background:var(--surface2); border:1px solid var(--border); border-radius:10px; padding:20px; }
.spec-card h4 { font-family:var(--mono); font-size:11px; text-transform:uppercase; letter-spacing:1px; color:var(--accent); margin-bottom:14px; display:flex; align-items:center; gap:8px; }
.spec-row { display:flex; justify-content:space-between; align-items:baseline; padding:5px 0; border-bottom:1px solid var(--border); gap:12px; }
.spec-row:last-child { border-bottom:none; }
.spec-key { font-size:12px; color:var(--muted); font-family:var(--mono); white-space:nowrap; }
.spec-val { font-size:12px; color:var(--text); font-family:var(--mono); text-align:right; word-break:break-all; }
.spec-val.accent { color:var(--accent); }
.spec-val.success { color:var(--success); }
.spec-val.teal { color:var(--teal); }
.spec-val.warning { color:var(--warning); }
.pkg-grid { display:grid; grid-template-columns:1fr 1fr; gap:4px 16px; }
.pkg-row { display:flex; justify-content:space-between; padding:4px 0; border-bottom:1px solid var(--border); }
.pkg-row:last-child { border-bottom:none; }
.pkg-name { font-family:var(--mono); font-size:12px; color:var(--teal); }
.pkg-ver  { font-family:var(--mono); font-size:12px; color:var(--muted); }
.model-row { display:flex; flex-direction:column; padding:6px 0; border-bottom:1px solid var(--border); gap:2px; }
.model-row:last-child { border-bottom:none; }
.model-key { font-family:var(--mono); font-size:12px; font-weight:700; color:var(--text); }
.model-desc { font-family:var(--mono); font-size:11px; color:var(--muted); }
.notes-box { background:var(--bg); border:1px solid var(--border); border-radius:6px; padding:12px 16px; font-family:var(--mono); font-size:12px; color:var(--muted); line-height:1.7; margin-top:8px; }
</style>
</head>
<body>
<div class="header">
  <div>
    <h1>OCR_BENCHMARK_DASHBOARD</h1>
    <p id="headerSub">Chargement...</p>
  </div>
  <span class="badge" id="headerBadge"></span>
</div>
<div class="main">
  <div class="filters-bar">
    <span class="filter-label">Parseur</span>
    <div id="parserToggles" style="display:flex;gap:6px;flex-wrap:wrap"></div>
    <span class="filter-label" style="margin-left:12px">Modèle OCR</span>
    <select id="ocrFilter"><option value="all">Tous</option></select>
    <span class="filter-label" style="margin-left:12px">Trier tableau par</span>
    <select id="sortBy">
      <option value="gt_accuracy_pct">GT Accuracy</option>
      <option value="fill_rate_pct">Fill Rate</option>
      <option value="llm_judge_score_pct">LLM Judge</option>
      <option value="avg_latency_s">Latence (ASC)</option>
    </select>
    <button class="btn btn-ghost" onclick="resetFilters()" style="margin-left:auto">↺ Reset</button>
  </div>
  <div id="kpiRow" class="kpi-row"></div>
  <div class="section-title">Précision &amp; Qualité</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>GT Accuracy par modèle OCR</h3><p class="chart-sub">% ground-truth · par parseur</p><div id="legAccuracy" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartAccuracy" role="img" aria-label="GT Accuracy">GT Accuracy</canvas></div></div>
    <div class="chart-card"><h3>Fill Rate par modèle OCR</h3><p class="chart-sub">% champs remplis · par parseur</p><div id="legFill" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartFill" role="img" aria-label="Fill Rate">Fill Rate</canvas></div></div>
  </div>
  <div class="section-title">Performance &amp; Latence</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>Latence moyenne (secondes)</h3><p class="chart-sub">avg_latency_s · plus bas = meilleur</p><div id="legLatency" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartLatency" role="img" aria-label="Latence">Latence</canvas></div></div>
    <div class="chart-card"><h3>LLM Judge Score</h3><p class="chart-sub">Score évaluation LLM juge · par parseur</p><div id="legJudge" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartJudge" role="img" aria-label="LLM Judge">LLM Judge</canvas></div></div>
  </div>
  <div class="section-title">Vue scatter — Accuracy vs Latence</div>
  <div class="chart-full">
    <h3>Accuracy vs Latence</h3>
    <p class="chart-sub">X = latence (s) · Y = GT Accuracy (%) · taille bulle = fill rate · idéal = haut-gauche</p>
    <div id="legScatter" class="legend"></div>
    <div style="position:relative;height:320px"><canvas id="chartScatter" role="img" aria-label="Scatter">Scatter</canvas></div>
  </div>
  <div class="section-title">Données détaillées</div>
  <div class="table-wrap">
    <div class="table-header"><h3>Tous les résultats</h3><span id="rowCount" style="font-size:12px;font-family:var(--mono);color:var(--muted)"></span></div>
    <div style="overflow-x:auto">
      <table><thead><tr>
        <th>#</th>
        <th onclick="sortTable('model')">Modèle ↕</th>
        <th onclick="sortTable('parser')">Parseur ↕</th>
        <th onclick="sortTable('gt_accuracy_pct')">GT Accuracy ↕</th>
        <th onclick="sortTable('fill_rate_pct')">Fill Rate ↕</th>
        <th onclick="sortTable('llm_judge_score_pct')">LLM Judge ↕</th>
        <th onclick="sortTable('avg_latency_s')">Latence (s) ↕</th>
      </tr></thead>
      <tbody id="tableBody"></tbody></table>
    </div>
  </div>
</div>

<!-- ── FOOTER SPECS ── -->
<footer class="footer">
  <button class="footer-toggle" id="footerToggle" onclick="toggleFooter()">
    <span>⚙ Spécifications techniques du benchmark</span>
    <span class="arrow">▾</span>
  </button>
  <div class="footer-body" id="footerBody"></div>
</footer>

<script>
const RAW_DATA = __DATA_JSON__;
const SPECS    = __SPECS_JSON__;
const COLORS = ['#4f9cf9','#7b6ef6','#2dd4bf','#fbbf24','#f87171','#4ade80','#fb923c','#e879f9'];
let parsers = [...new Set(RAW_DATA.map(r=>r.parser))];
let activeParsers = new Set(parsers);
let ocrFilter = 'all';
let tableSortCol = 'gt_accuracy_pct', tableSortDir = -1;
const charts = {};

function col(p,a=1){const h=COLORS[parsers.indexOf(p)%COLORS.length];if(a===1)return h;const r=parseInt(h.slice(1,3),16),g=parseInt(h.slice(3,5),16),b=parseInt(h.slice(5,7),16);return `rgba(${r},${g},${b},${a})`;}
function fmt(v){return v==null?'—':typeof v==='number'?v.toFixed(1):v;}
function filtered(){return RAW_DATA.filter(r=>activeParsers.has(r.parser)&&(ocrFilter==='all'||r.model===ocrFilter));}

function buildToggles(){
  document.getElementById('parserToggles').innerHTML=parsers.map((p,i)=>`
    <button class="parser-toggle active" data-i="${i}" onclick="toggleP('${p}',this,${i})"
      style="background:${COLORS[i]};border-color:${COLORS[i]};color:#0d0f14">${p}</button>`).join('');
}
function toggleP(p,btn,i){
  if(activeParsers.has(p)){if(activeParsers.size===1)return;activeParsers.delete(p);btn.classList.remove('active');btn.style.cssText=`background:transparent;color:var(--muted);border-color:var(--border2)`;}
  else{activeParsers.add(p);btn.classList.add('active');btn.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;}
  updateAll();
}
function buildOcrFilter(){
  const models=[...new Set(RAW_DATA.map(r=>r.model))].sort();
  const sel=document.getElementById('ocrFilter');
  sel.innerHTML='<option value="all">Tous</option>'+models.map(m=>`<option value="${m}">${m}</option>`).join('');
  sel.onchange=()=>{ocrFilter=sel.value;updateAll();};
}
function resetFilters(){
  activeParsers=new Set(parsers);ocrFilter='all';
  document.getElementById('ocrFilter').value='all';
  document.querySelectorAll('.parser-toggle').forEach((b,i)=>{b.classList.add('active');b.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;});
  updateAll();
}
document.getElementById('sortBy').onchange=function(){tableSortCol=this.value;tableSortDir=this.value==='avg_latency_s'?1:-1;buildTable(filtered());};

function buildKPIs(data){
  if(!data.length)return;
  const best=data.reduce((a,b)=>(b.gt_accuracy_pct||0)>(a.gt_accuracy_pct||0)?b:a);
  const avg=k=>(data.reduce((s,r)=>s+(r[k]||0),0)/data.length).toFixed(1);
  document.getElementById('kpiRow').innerHTML=`
    <div class="kpi-card c-green"><div class="label">Meilleur GT Accuracy</div><div class="value">${fmt(best.gt_accuracy_pct)}%</div><div class="sub">${best.model} · ${best.parser}</div></div>
    <div class="kpi-card c-blue"><div class="label">Accuracy moyenne</div><div class="value">${avg('gt_accuracy_pct')}%</div><div class="sub">${data.length} entrées filtrées</div></div>
    <div class="kpi-card c-teal"><div class="label">Fill Rate moyen</div><div class="value">${avg('fill_rate_pct')}%</div><div class="sub">champs remplis</div></div>
    <div class="kpi-card c-purple"><div class="label">Latence moyenne</div><div class="value">${avg('avg_latency_s')}s</div><div class="sub">avg_latency_s</div></div>`;
}

function legend(id,pArr){document.getElementById(id).innerHTML=pArr.map(p=>`<span class="legend-item"><span class="legend-dot" style="background:${col(p)}"></span>${p}</span>`).join('');}

const tick={color:'#6b7080',font:{size:11,family:"'Space Mono',monospace"}};
const grid={color:'rgba(255,255,255,0.05)'};
const baseOpts={responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{bodyFont:{family:"'Space Mono',monospace",size:12}}},scales:{x:{ticks:{...tick,maxRotation:35},grid},y:{ticks:tick,grid}}};

function buildCharts(data){
  const activePArr=[...activeParsers].filter(p=>data.some(r=>r.parser===p));
  const models=[...new Set(data.map(r=>r.model))].sort();
  function ds(metric){return activePArr.map(p=>({label:p,data:models.map(m=>{const r=data.find(d=>d.model===m&&d.parser===p);return r?(r[metric]??null):null;}),backgroundColor:col(p,.75),borderColor:col(p),borderWidth:1.5,borderRadius:3}));}
  ['chartAccuracy','chartFill','chartLatency','chartJudge','chartScatter'].forEach(id=>{if(charts[id]){charts[id].destroy();delete charts[id];}});
  charts.chartAccuracy=new Chart(document.getElementById('chartAccuracy'),{type:'bar',data:{labels:models,datasets:ds('gt_accuracy_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legAccuracy',activePArr);
  charts.chartFill=new Chart(document.getElementById('chartFill'),{type:'bar',data:{labels:models,datasets:ds('fill_rate_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legFill',activePArr);
  charts.chartLatency=new Chart(document.getElementById('chartLatency'),{type:'bar',data:{labels:models,datasets:ds('avg_latency_s')},options:baseOpts});
  legend('legLatency',activePArr);
  charts.chartJudge=new Chart(document.getElementById('chartJudge'),{type:'bar',data:{labels:models,datasets:ds('llm_judge_score_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legJudge',activePArr);
  charts.chartScatter=new Chart(document.getElementById('chartScatter'),{type:'bubble',
    data:{datasets:activePArr.map(p=>({label:p,data:data.filter(r=>r.parser===p).map(r=>({x:r.avg_latency_s,y:r.gt_accuracy_pct,r:Math.max(5,(r.fill_rate_pct/100)*18),model:r.model,fill:r.fill_rate_pct})),backgroundColor:col(p,.5),borderColor:col(p),borderWidth:1.5}))},
    options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>`${c.raw.model} (${c.dataset.label}) acc:${c.raw.y}% lat:${c.raw.x}s fill:${c.raw.fill}%`},bodyFont:{family:"'Space Mono',monospace",size:11}}},
    scales:{x:{title:{display:true,text:'Latence (s)',color:'#6b7080'},ticks:tick,grid},y:{title:{display:true,text:'GT Accuracy (%)',color:'#6b7080'},ticks:tick,grid,min:0,max:105}}}});
  legend('legScatter',activePArr);
}

function sortTable(c){if(tableSortCol===c)tableSortDir*=-1;else{tableSortCol=c;tableSortDir=-1;}buildTable(filtered());}
function buildTable(data){
  const sorted=[...data].sort((a,b)=>{const av=a[tableSortCol]??'',bv=b[tableSortCol]??'';return(typeof av==='number'?(av-bv):String(av).localeCompare(String(bv)))*tableSortDir;});
  document.getElementById('rowCount').textContent=`${sorted.length} lignes`;
  const maxV=k=>Math.max(...data.map(r=>r[k]||0));
  const [mA,mF,mJ,mL]=['gt_accuracy_pct','fill_rate_pct','llm_judge_score_pct','avg_latency_s'].map(maxV);
  function bar(v,max,c){const p=max?(v/max*100):0;return `<div class="bar-cell"><span class="td-val">${fmt(v)}</span><div class="bar-track"><div class="bar-fill" style="width:${p}%;background:${c}"></div></div></div>`;}
  document.getElementById('tableBody').innerHTML=sorted.map((r,i)=>{
    const pi=parsers.indexOf(r.parser),c=COLORS[pi%COLORS.length],rc=i<3?`rank-${i+1}`:'rank-other';
    return `<tr><td><span class="rank-badge ${rc}">${i+1}</span></td><td class="td-model">${r.model}</td><td><span class="td-parser" style="background:${c}22;color:${c};border:1px solid ${c}44">${r.parser}</span></td><td>${bar(r.gt_accuracy_pct,mA,'#4ade80')}</td><td>${bar(r.fill_rate_pct,mF,'#4f9cf9')}</td><td>${bar(r.llm_judge_score_pct,mJ,'#7b6ef6')}</td><td>${bar(r.avg_latency_s,mL,'#fbbf24')}</td></tr>`;
  }).join('');
}
function updateAll(){const d=filtered();buildKPIs(d);buildCharts(d);buildTable(d);}

// ── FOOTER ──────────────────────────────────────────────────────────────
function toggleFooter(){
  const btn=document.getElementById('footerToggle');
  const body=document.getElementById('footerBody');
  btn.classList.toggle('open');
  body.classList.toggle('open');
}

function buildFooter(){
  const s = SPECS;
  if(!s || !Object.keys(s).length) return;

  function row(k,v,cls=''){return `<div class="spec-row"><span class="spec-key">${k}</span><span class="spec-val ${cls}">${v||'—'}</span></div>`;}

  // Card 1 — Environnement
  const envFields = [
    ['date',s.date,'accent'],['os',s.os,''],['python',s.python_version,'teal'],
    ['cpu',s.cpu,''],['gpu',s.gpu,'warning'],['ram',s.ram,''],
    ['pipeline',s.pipeline_version,'accent']
  ].filter(([,v])=>v);

  // Card 2 — Dataset & évaluation
  const evalFields = [
    ['dataset',s.dataset,''],['split',s.dataset_split,''],
    ['métriques',s.eval_metric,'teal'],['runs',s.n_runs,'']
  ].filter(([,v])=>v);

  // Card 3 — Packages
  const pkgs = s.packages ? Object.entries(s.packages) : [];

  // Card 4 — Modèles OCR
  const ocrModels = s.ocr_models ? Object.entries(s.ocr_models) : [];

  // Card 5 — Parseurs LLM
  const llmParsers = s.llm_parsers ? Object.entries(s.llm_parsers) : [];

  let html = '<div class="specs-grid">';

  // Env card
  if(envFields.length){
    html += `<div class="spec-card"><h4>🖥 Environnement</h4>${envFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  }

  // Eval card
  if(evalFields.length){
    html += `<div class="spec-card"><h4>📐 Dataset &amp; Évaluation</h4>${evalFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  }

  // Packages card
  if(pkgs.length){
    const pkgRows = pkgs.map(([n,v])=>`<div class="pkg-row"><span class="pkg-name">${n}</span><span class="pkg-ver">${v}</span></div>`).join('');
    html += `<div class="spec-card"><h4>📦 Packages Python</h4><div class="pkg-grid">${pkgRows}</div></div>`;
  }

  // OCR models card
  if(ocrModels.length){
    const mRows = ocrModels.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('');
    html += `<div class="spec-card"><h4>🔍 Modèles OCR testés</h4>${mRows}</div>`;
  }

  // LLM parsers card
  if(llmParsers.length){
    const pRows = llmParsers.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('');
    html += `<div class="spec-card"><h4>🤖 Parseurs LLM testés</h4>${pRows}</div>`;
  }

  html += '</div>';

  // Notes
  if(s.notes){
    html += `<div class="notes-box">📝 &nbsp;${s.notes}</div>`;
  }

  document.getElementById('footerBody').innerHTML = html;
}

// Init
buildToggles();
buildOcrFilter();
document.getElementById('headerSub').textContent=`// ${parsers.length} parseur(s) · ${RAW_DATA.length} entrées`;
document.getElementById('headerBadge').textContent=`${parsers.length} parseurs chargés`;
updateAll();
buildFooter();
</script>
</body></html>
"""

generate_dashboard(df, OUTPUT_HTML, BENCHMARK_SPECS)

In [ ]:
def generate_dashboard(df: pd.DataFrame, output_path: str, specs: dict = None):
    records = df.where(pd.notnull(df), None).to_dict(orient='records')
    data_json  = json.dumps(records, ensure_ascii=False)
    specs_json = json.dumps(specs or {}, ensure_ascii=False)

    html = HTML_TEMPLATE.replace('__DATA_JSON__', data_json).replace('__SPECS_JSON__', specs_json)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(html)

    print(f'✅ Dashboard généré : {Path(output_path).resolve()}')
    print(f'   {len(records)} entrées · {df["parser"].nunique()} parseur(s)')


HTML_TEMPLATE = r"""
<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>OCR Benchmark Dashboard</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap');
*, *::before, *::after { box-sizing: border-box; margin: 0; padding: 0; }
:root {
  --bg: #0d0f14; --surface: #141720; --surface2: #1c2030;
  --border: rgba(255,255,255,0.07); --border2: rgba(255,255,255,0.14);
  --text: #e8eaf0; --muted: #6b7080;
  --accent: #4f9cf9; --accent2: #7b6ef6;
  --success: #4ade80; --warning: #fbbf24; --danger: #f87171; --teal: #2dd4bf;
  --mono: 'Space Mono', monospace; --sans: 'DM Sans', sans-serif;
}
body { background: var(--bg); color: var(--text); font-family: var(--sans); font-size: 14px; min-height: 100vh; }
.header { border-bottom:1px solid var(--border); padding:20px 32px; display:flex; align-items:center; justify-content:space-between; background:var(--surface); position:sticky; top:0; z-index:10; }
.header h1 { font-family:var(--mono); font-size:17px; font-weight:700; }
.header p { color:var(--muted); font-size:11px; margin-top:3px; font-family:var(--mono); }
.badge { background:rgba(79,156,249,0.12); color:var(--accent); border:1px solid rgba(79,156,249,0.25); padding:4px 10px; border-radius:4px; font-family:var(--mono); font-size:11px; }
.main { max-width:1400px; margin:0 auto; padding:28px 32px; }
.filters-bar { display:flex; gap:12px; margin-bottom:24px; flex-wrap:wrap; align-items:center; background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:14px 18px; }
.filter-label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; }
.parser-toggle { padding:5px 12px; border-radius:20px; border:1px solid var(--border2); background:transparent; color:var(--muted); font-size:12px; font-family:var(--mono); cursor:pointer; transition:all .15s; }
.parser-toggle.active { color:#0d0f14; border-color:transparent; font-weight:700; }
select { background:var(--surface); border:1px solid var(--border2); color:var(--text); padding:7px 12px; border-radius:6px; font-family:var(--sans); font-size:13px; cursor:pointer; outline:none; }
select:focus { border-color:var(--accent); }
.btn { background:var(--accent); color:#0d0f14; border:none; padding:8px 16px; border-radius:6px; font-family:var(--sans); font-size:13px; font-weight:600; cursor:pointer; transition:opacity .15s; }
.btn:hover { opacity:.85; }
.btn-ghost { background:transparent; color:var(--text); border:1px solid var(--border2); }
.btn-ghost:hover { background:var(--surface2); }
.kpi-row { display:grid; grid-template-columns:repeat(auto-fit,minmax(200px,1fr)); gap:16px; margin-bottom:32px; }
.kpi-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; position:relative; overflow:hidden; }
.kpi-card::before { content:''; position:absolute; top:0; left:0; right:0; height:2px; }
.kpi-card.c-blue::before { background:var(--accent); } .kpi-card.c-green::before { background:var(--success); }
.kpi-card.c-purple::before { background:var(--accent2); } .kpi-card.c-teal::before { background:var(--teal); }
.kpi-card.c-red::before { background:var(--danger); }
.kpi-card .label { font-size:11px; font-family:var(--mono); color:var(--muted); text-transform:uppercase; letter-spacing:.5px; margin-bottom:10px; }
.kpi-card .value { font-size:28px; font-weight:600; font-family:var(--mono); line-height:1; }
.kpi-card .sub { font-size:11px; color:var(--muted); margin-top:6px; font-family:var(--mono); }
.kpi-card.c-blue .value{color:var(--accent)} .kpi-card.c-green .value{color:var(--success)} .kpi-card.c-purple .value{color:var(--accent2)} .kpi-card.c-teal .value{color:var(--teal)} .kpi-card.c-red .value{color:var(--danger)}
.section-title { font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; color:var(--muted); margin-bottom:16px; display:flex; align-items:center; gap:10px; }
.section-title::after { content:''; flex:1; height:1px; background:var(--border); }
.chart-grid { display:grid; grid-template-columns:1fr 1fr; gap:20px; margin-bottom:32px; }
@media(max-width:900px){.chart-grid{grid-template-columns:1fr;}}
.chart-card { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; }
.chart-card h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-card .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.chart-full { background:var(--surface); border:1px solid var(--border); border-radius:10px; padding:20px; margin-bottom:32px; }
.chart-full h3 { font-size:13px; font-weight:500; font-family:var(--mono); margin-bottom:4px; }
.chart-full .chart-sub { font-size:11px; color:var(--muted); margin-bottom:12px; font-family:var(--mono); }
.legend { display:flex; flex-wrap:wrap; gap:14px; margin-bottom:10px; }
.legend-item { display:flex; align-items:center; gap:6px; font-size:12px; font-family:var(--mono); color:var(--muted); }
.legend-dot { width:10px; height:10px; border-radius:2px; }
.table-wrap { background:var(--surface); border:1px solid var(--border); border-radius:10px; overflow:hidden; margin-bottom:32px; }
.table-header { padding:16px 20px; border-bottom:1px solid var(--border); display:flex; align-items:center; justify-content:space-between; }
.table-header h3 { font-family:var(--mono); font-size:13px; font-weight:500; }
table { width:100%; border-collapse:collapse; }
thead th { padding:10px 16px; text-align:left; font-family:var(--mono); font-size:11px; color:var(--muted); text-transform:uppercase; letter-spacing:.5px; background:var(--surface2); cursor:pointer; user-select:none; white-space:nowrap; }
thead th:hover { color:var(--text); }
tbody tr { border-top:1px solid var(--border); transition:background .1s; }
tbody tr:hover { background:var(--surface2); }
tbody td { padding:11px 16px; font-size:13px; white-space:nowrap; }
.td-model { font-family:var(--mono); font-weight:700; }
.td-parser { font-family:var(--mono); font-size:11px; padding:3px 8px; border-radius:4px; }
.bar-cell { display:flex; align-items:center; gap:10px; }
.bar-track { flex:1; max-width:90px; height:4px; background:var(--surface2); border-radius:2px; overflow:hidden; }
.bar-fill { height:100%; border-radius:2px; }
.td-val { font-family:var(--mono); font-size:13px; min-width:45px; }
.rank-badge { display:inline-flex; align-items:center; justify-content:center; width:22px; height:22px; border-radius:4px; font-family:var(--mono); font-size:11px; font-weight:700; }
.rank-1{background:rgba(251,191,36,.15);color:var(--warning);border:1px solid rgba(251,191,36,.3)}
.rank-2{background:rgba(148,163,184,.1);color:#94a3b8;border:1px solid rgba(148,163,184,.2)}
.rank-3{background:rgba(180,130,80,.1);color:#b48250;border:1px solid rgba(180,130,80,.2)}
.rank-other{background:var(--surface2);color:var(--muted);border:1px solid var(--border)}
.footer { border-top:1px solid var(--border); background:var(--surface); margin-top:16px; }
.footer-toggle { width:100%; display:flex; align-items:center; justify-content:space-between; padding:16px 32px; background:transparent; border:none; color:var(--muted); font-family:var(--mono); font-size:12px; text-transform:uppercase; letter-spacing:1px; cursor:pointer; transition:color .15s; }
.footer-toggle:hover { color:var(--text); }
.footer-toggle .arrow { transition:transform .25s; font-size:14px; }
.footer-toggle.open .arrow { transform:rotate(180deg); }
.footer-body { display:none; padding:0 32px 32px; max-width:1400px; margin:0 auto; }
.footer-body.open { display:block; }
.specs-grid { display:grid; grid-template-columns:repeat(auto-fit,minmax(280px,1fr)); gap:20px; margin-top:8px; }
.spec-card { background:var(--surface2); border:1px solid var(--border); border-radius:10px; padding:20px; }
.spec-card h4 { font-family:var(--mono); font-size:11px; text-transform:uppercase; letter-spacing:1px; color:var(--accent); margin-bottom:14px; display:flex; align-items:center; gap:8px; }
.spec-row { display:flex; justify-content:space-between; align-items:baseline; padding:5px 0; border-bottom:1px solid var(--border); gap:12px; }
.spec-row:last-child { border-bottom:none; }
.spec-key { font-size:12px; color:var(--muted); font-family:var(--mono); white-space:nowrap; }
.spec-val { font-size:12px; color:var(--text); font-family:var(--mono); text-align:right; word-break:break-all; }
.spec-val.accent { color:var(--accent); }
.spec-val.success { color:var(--success); }
.spec-val.teal { color:var(--teal); }
.spec-val.warning { color:var(--warning); }
.pkg-grid { display:grid; grid-template-columns:1fr 1fr; gap:4px 16px; }
.pkg-row { display:flex; justify-content:space-between; padding:4px 0; border-bottom:1px solid var(--border); }
.pkg-row:last-child { border-bottom:none; }
.pkg-name { font-family:var(--mono); font-size:12px; color:var(--teal); }
.pkg-ver  { font-family:var(--mono); font-size:12px; color:var(--muted); }
.model-row { display:flex; flex-direction:column; padding:6px 0; border-bottom:1px solid var(--border); gap:2px; }
.model-row:last-child { border-bottom:none; }
.model-key { font-family:var(--mono); font-size:12px; font-weight:700; color:var(--text); }
.model-desc { font-family:var(--mono); font-size:11px; color:var(--muted); }
.notes-box { background:var(--bg); border:1px solid var(--border); border-radius:6px; padding:12px 16px; font-family:var(--mono); font-size:12px; color:var(--muted); line-height:1.7; margin-top:8px; }
</style>
</head>
<body>
<div class="header">
  <div>
    <h1>OCR_BENCHMARK_DASHBOARD</h1>
    <p id="headerSub">Chargement...</p>
  </div>
  <span class="badge" id="headerBadge"></span>
</div>
<div class="main">
  <div class="filters-bar">
    <span class="filter-label">Parseur</span>
    <div id="parserToggles" style="display:flex;gap:6px;flex-wrap:wrap"></div>
    <span class="filter-label" style="margin-left:12px">Modèle OCR</span>
    <select id="ocrFilter"><option value="all">Tous</option></select>
    <span class="filter-label" style="margin-left:12px">Trier tableau par</span>
    <select id="sortBy">
      <option value="gt_accuracy_pct">GT Accuracy</option>
      <option value="fill_rate_pct">Fill Rate</option>
      <option value="avg_latency_s">Latence (ASC)</option>
    </select>
    <button class="btn btn-ghost" onclick="resetFilters()" style="margin-left:auto">↺ Reset</button>
  </div>
  <div id="kpiRow" class="kpi-row"></div>
  <div class="section-title">Précision &amp; Qualité</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>GT Accuracy par modèle OCR</h3><p class="chart-sub">% ground-truth · par parseur</p><div id="legAccuracy" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartAccuracy"></canvas></div></div>
    <div class="chart-card"><h3>Fill Rate par modèle OCR</h3><p class="chart-sub">% champs remplis · par parseur</p><div id="legFill" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartFill"></canvas></div></div>
  </div>
  <div class="section-title">Performance &amp; Latence</div>
  <div class="chart-grid">
    <div class="chart-card"><h3>Latence moyenne (secondes)</h3><p class="chart-sub">avg_latency_s · plus bas = meilleur</p><div id="legLatency" class="legend"></div><div style="position:relative;height:260px"><canvas id="chartLatency"></canvas></div></div>
  </div>
  <div class="section-title">Vue scatter — Accuracy vs Latence</div>
  <div class="chart-full">
    <h3>Accuracy vs Latence</h3>
    <p class="chart-sub">X = latence (s) · Y = GT Accuracy (%) · taille bulle = fill rate · idéal = haut-gauche</p>
    <div id="legScatter" class="legend"></div>
    <div style="position:relative;height:320px"><canvas id="chartScatter"></canvas></div>
  </div>
  <div class="section-title">Données détaillées</div>
  <div class="table-wrap">
    <div class="table-header"><h3>Tous les résultats</h3><span id="rowCount" style="font-size:12px;font-family:var(--mono);color:var(--muted)"></span></div>
    <div style="overflow-x:auto">
      <table><thead><tr>
        <th>#</th>
        <th onclick="sortTable('model')">Modèle ↕</th>
        <th onclick="sortTable('parser')">Parseur ↕</th>
        <th onclick="sortTable('gt_accuracy_pct')">GT Accuracy ↕</th>
        <th onclick="sortTable('fill_rate_pct')">Fill Rate ↕</th>
        <th onclick="sortTable('avg_latency_s')">Latence (s) ↕</th>
      </tr></thead>
      <tbody id="tableBody"></tbody></table>
    </div>
  </div>
</div>

<footer class="footer">
  <button class="footer-toggle" id="footerToggle" onclick="toggleFooter()">
    <span>⚙ Spécifications techniques du benchmark</span>
    <span class="arrow">▾</span>
  </button>
  <div class="footer-body" id="footerBody"></div>
</footer>

<script>
const RAW_DATA = __DATA_JSON__;
const SPECS    = __SPECS_JSON__;
const COLORS = ['#4f9cf9','#7b6ef6','#2dd4bf','#fbbf24','#f87171','#4ade80','#fb923c','#e879f9'];
let parsers = [...new Set(RAW_DATA.map(r=>r.parser))];
let activeParsers = new Set(parsers);
let ocrFilter = 'all';
let tableSortCol = 'gt_accuracy_pct', tableSortDir = -1;
const charts = {};

function col(p,a=1){const h=COLORS[parsers.indexOf(p)%COLORS.length];if(a===1)return h;const r=parseInt(h.slice(1,3),16),g=parseInt(h.slice(3,5),16),b=parseInt(h.slice(5,7),16);return `rgba(${r},${g},${b},${a})`;}
function fmt(v){return v==null?'—':typeof v==='number'?v.toFixed(1):v;}
function filtered(){return RAW_DATA.filter(r=>activeParsers.has(r.parser)&&(ocrFilter==='all'||r.model===ocrFilter));}

function buildToggles(){
  document.getElementById('parserToggles').innerHTML=parsers.map((p,i)=>`
    <button class="parser-toggle active" data-i="${i}" onclick="toggleP('${p}',this,${i})"
      style="background:${COLORS[i]};border-color:${COLORS[i]};color:#0d0f14">${p}</button>`).join('');
}
function toggleP(p,btn,i){
  if(activeParsers.has(p)){if(activeParsers.size===1)return;activeParsers.delete(p);btn.classList.remove('active');btn.style.cssText=`background:transparent;color:var(--muted);border-color:var(--border2)`;}
  else{activeParsers.add(p);btn.classList.add('active');btn.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;}
  updateAll();
}
function buildOcrFilter(){
  const models=[...new Set(RAW_DATA.map(r=>r.model))].sort();
  const sel=document.getElementById('ocrFilter');
  sel.innerHTML='<option value="all">Tous</option>'+models.map(m=>`<option value="${m}">${m}</option>`).join('');
  sel.onchange=()=>{ocrFilter=sel.value;updateAll();};
}
function resetFilters(){
  activeParsers=new Set(parsers);ocrFilter='all';
  document.getElementById('ocrFilter').value='all';
  document.querySelectorAll('.parser-toggle').forEach((b,i)=>{b.classList.add('active');b.style.cssText=`background:${COLORS[i]};color:#0d0f14;border-color:${COLORS[i]}`;});
  updateAll();
}
document.getElementById('sortBy').onchange=function(){
  tableSortCol=this.value;
  tableSortDir=this.value==='avg_latency_s'?1:-1;
  buildTable(filtered());
};

function buildKPIs(data){
  if(!data.length)return;

  // KPI 1 — Meilleur GT Accuracy
  const bestAcc = data.reduce((a,b)=>(b.gt_accuracy_pct||0)>(a.gt_accuracy_pct||0)?b:a);

  // KPI 2 — Meilleur Fill Rate
  const bestFill = data.reduce((a,b)=>(b.fill_rate_pct||0)>(a.fill_rate_pct||0)?b:a);

  // KPI 3 — Modèle le plus rapide (latence min)
  const fastest = data.reduce((a,b)=>(b.avg_latency_s!=null&&(a.avg_latency_s==null||b.avg_latency_s<a.avg_latency_s))?b:a);

  // KPI 4 — Pire latence (à éviter)
  const slowest = data.reduce((a,b)=>(b.avg_latency_s!=null&&(a.avg_latency_s==null||b.avg_latency_s>a.avg_latency_s))?b:a);

  document.getElementById('kpiRow').innerHTML=`
    <div class="kpi-card c-green">
      <div class="label">🏆 Meilleur GT Accuracy</div>
      <div class="value">${fmt(bestAcc.gt_accuracy_pct)}%</div>
      <div class="sub">${bestAcc.model} · ${bestAcc.parser}</div>
    </div>
    <div class="kpi-card c-blue">
      <div class="label">📋 Meilleur Fill Rate</div>
      <div class="value">${fmt(bestFill.fill_rate_pct)}%</div>
      <div class="sub">${bestFill.model} · ${bestFill.parser}</div>
    </div>
    <div class="kpi-card c-teal">
      <div class="label">⚡ Plus rapide</div>
      <div class="value">${fmt(fastest.avg_latency_s)}s</div>
      <div class="sub">${fastest.model} · ${fastest.parser}</div>
    </div>
    <div class="kpi-card c-red">
      <div class="label">🐢 Pire latence</div>
      <div class="value">${fmt(slowest.avg_latency_s)}s</div>
      <div class="sub">${slowest.model} · ${slowest.parser}</div>
    </div>`;
}

function legend(id,pArr){document.getElementById(id).innerHTML=pArr.map(p=>`<span class="legend-item"><span class="legend-dot" style="background:${col(p)}"></span>${p}</span>`).join('');}

const tick={color:'#6b7080',font:{size:11,family:"'Space Mono',monospace"}};
const grid={color:'rgba(255,255,255,0.05)'};
const baseOpts={responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{bodyFont:{family:"'Space Mono',monospace",size:12}}},scales:{x:{ticks:{...tick,maxRotation:35},grid},y:{ticks:tick,grid}}};

function buildCharts(data){
  const activePArr=[...activeParsers].filter(p=>data.some(r=>r.parser===p));
  const models=[...new Set(data.map(r=>r.model))].sort();
  function ds(metric){return activePArr.map(p=>({label:p,data:models.map(m=>{const r=data.find(d=>d.model===m&&d.parser===p);return r?(r[metric]??null):null;}),backgroundColor:col(p,.75),borderColor:col(p),borderWidth:1.5,borderRadius:3}));}
  ['chartAccuracy','chartFill','chartLatency','chartScatter'].forEach(id=>{if(charts[id]){charts[id].destroy();delete charts[id];}});
  charts.chartAccuracy=new Chart(document.getElementById('chartAccuracy'),{type:'bar',data:{labels:models,datasets:ds('gt_accuracy_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legAccuracy',activePArr);
  charts.chartFill=new Chart(document.getElementById('chartFill'),{type:'bar',data:{labels:models,datasets:ds('fill_rate_pct')},options:{...baseOpts,scales:{...baseOpts.scales,y:{...baseOpts.scales.y,max:100}}}});
  legend('legFill',activePArr);
  charts.chartLatency=new Chart(document.getElementById('chartLatency'),{type:'bar',data:{labels:models,datasets:ds('avg_latency_s')},options:baseOpts});
  legend('legLatency',activePArr);
  charts.chartScatter=new Chart(document.getElementById('chartScatter'),{type:'bubble',
    data:{datasets:activePArr.map(p=>({label:p,data:data.filter(r=>r.parser===p).map(r=>({x:r.avg_latency_s,y:r.gt_accuracy_pct,r:Math.max(5,(r.fill_rate_pct/100)*18),model:r.model,fill:r.fill_rate_pct})),backgroundColor:col(p,.5),borderColor:col(p),borderWidth:1.5}))},
    options:{responsive:true,maintainAspectRatio:false,plugins:{legend:{display:false},tooltip:{callbacks:{label:c=>`${c.raw.model} (${c.dataset.label}) acc:${c.raw.y}% lat:${c.raw.x}s fill:${c.raw.fill}%`},bodyFont:{family:"'Space Mono',monospace",size:11}}},
    scales:{x:{title:{display:true,text:'Latence (s)',color:'#6b7080'},ticks:tick,grid},y:{title:{display:true,text:'GT Accuracy (%)',color:'#6b7080'},ticks:tick,grid,min:0,max:105}}}});
  legend('legScatter',activePArr);
}

function sortTable(c){if(tableSortCol===c)tableSortDir*=-1;else{tableSortCol=c;tableSortDir=-1;}buildTable(filtered());}
function buildTable(data){
  const sorted=[...data].sort((a,b)=>{const av=a[tableSortCol]??'',bv=b[tableSortCol]??'';return(typeof av==='number'?(av-bv):String(av).localeCompare(String(bv)))*tableSortDir;});
  document.getElementById('rowCount').textContent=`${sorted.length} lignes`;
  const maxV=k=>Math.max(...data.map(r=>r[k]||0));
  const [mA,mF,mL]=['gt_accuracy_pct','fill_rate_pct','avg_latency_s'].map(maxV);
  function bar(v,max,c){const p=max?(v/max*100):0;return `<div class="bar-cell"><span class="td-val">${fmt(v)}</span><div class="bar-track"><div class="bar-fill" style="width:${p}%;background:${c}"></div></div></div>`;}
  document.getElementById('tableBody').innerHTML=sorted.map((r,i)=>{
    const pi=parsers.indexOf(r.parser),c=COLORS[pi%COLORS.length],rc=i<3?`rank-${i+1}`:'rank-other';
    return `<tr><td><span class="rank-badge ${rc}">${i+1}</span></td><td class="td-model">${r.model}</td><td><span class="td-parser" style="background:${c}22;color:${c};border:1px solid ${c}44">${r.parser}</span></td><td>${bar(r.gt_accuracy_pct,mA,'#4ade80')}</td><td>${bar(r.fill_rate_pct,mF,'#4f9cf9')}</td><td>${bar(r.avg_latency_s,mL,'#fbbf24')}</td></tr>`;
  }).join('');
}
function updateAll(){const d=filtered();buildKPIs(d);buildCharts(d);buildTable(d);}

function toggleFooter(){
  const btn=document.getElementById('footerToggle');
  const body=document.getElementById('footerBody');
  btn.classList.toggle('open');
  body.classList.toggle('open');
}

function buildFooter(){
  const s = SPECS;
  if(!s || !Object.keys(s).length) return;
  function row(k,v,cls=''){return `<div class="spec-row"><span class="spec-key">${k}</span><span class="spec-val ${cls}">${v||'—'}</span></div>`;}
  const envFields=[['date',s.date,'accent'],['os',s.os,''],['python',s.python_version,'teal'],['cpu',s.cpu,''],['gpu',s.gpu,'warning'],['ram',s.ram,''],['pipeline',s.pipeline_version,'accent']].filter(([,v])=>v);
  const evalFields=[['dataset',s.dataset,''],['split',s.dataset_split,''],['métriques',s.eval_metric,'teal'],['runs',s.n_runs,'']].filter(([,v])=>v);
  const pkgs=s.packages?Object.entries(s.packages):[];
  const ocrModels=s.ocr_models?Object.entries(s.ocr_models):[];
  const llmParsers=s.llm_parsers?Object.entries(s.llm_parsers):[];
  let html='<div class="specs-grid">';
  if(envFields.length)html+=`<div class="spec-card"><h4>🖥 Environnement</h4>${envFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  if(evalFields.length)html+=`<div class="spec-card"><h4>📐 Dataset &amp; Évaluation</h4>${evalFields.map(([k,v,c])=>row(k,v,c)).join('')}</div>`;
  if(pkgs.length)html+=`<div class="spec-card"><h4>📦 Packages Python</h4><div class="pkg-grid">${pkgs.map(([n,v])=>`<div class="pkg-row"><span class="pkg-name">${n}</span><span class="pkg-ver">${v}</span></div>`).join('')}</div></div>`;
  if(ocrModels.length)html+=`<div class="spec-card"><h4>🔍 Modèles OCR testés</h4>${ocrModels.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('')}</div>`;
  if(llmParsers.length)html+=`<div class="spec-card"><h4>🤖 Parseurs LLM testés</h4>${llmParsers.map(([k,v])=>`<div class="model-row"><span class="model-key">${k}</span><span class="model-desc">${v}</span></div>`).join('')}</div>`;
  html+='</div>';
  if(s.notes)html+=`<div class="notes-box">📝 &nbsp;${s.notes}</div>`;
  document.getElementById('footerBody').innerHTML=html;
}

buildToggles();
buildOcrFilter();
document.getElementById('headerSub').textContent=`// ${parsers.length} parseur(s) · ${RAW_DATA.length} entrées`;
document.getElementById('headerBadge').textContent=`${parsers.length} parseurs chargés`;
updateAll();
buildFooter();
</script>
</body></html>
"""

generate_dashboard(df, OUTPUT_HTML, BENCHMARK_SPECS)

## 🚀 Ouvrir le dashboard

In [ ]:
from IPython.display import IFrame, display, HTML

# Affichage inline dans le notebook
display(HTML(f'<a href="{OUTPUT_HTML}" target="_blank" style="font-size:15px;font-weight:bold">🌐 Ouvrir le dashboard dans un nouvel onglet</a>'))

# Aperçu inline (peut être limité selon votre env Jupyter)
IFrame(src=OUTPUT_HTML, width='100%', height=900)

In [ ]:
"""
Rapport final — Comparaison parseurs × modèles OCR.

Lit tous les dossiers results_benchmark_*/ au même niveau,
extrait les métriques de chaque combinaison parseur+modèle
et produit deux CSV :

  1. report_global.csv     — une ligne par combinaison parseur+modèle
  2. report_by_field.csv   — une ligne par combinaison parseur+modèle+champ

Structure attendue :
    results_benchmark_regex/
        raw/
            paddleocr/
                carte_001.json
                carte_002.json
                ...
            markitdown/
                ...
    results_benchmark_mistral/
        raw/
            paddleocr/
                ...

Usage :
    python phase4_report.py
    python phase4_report.py --root /chemin/vers/dossier/parent
    python phase4_report.py --root . --prefix results_benchmark
"""

import argparse
import csv
import json
from collections import defaultdict
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
# Adapte si besoin
DEFAULT_ROOT   = Path(".")          # dossier parent qui contient les results_benchmark_*/
BENCHMARK_PREFIX = "results_benchmark_"
OUTPUT_DIR     = Path("report_final")

# Champs à évaluer — remplace par ton ALL_VARS réel
try:
    from config import ALL_VARS, GROUND_TRUTH_DIR
except ImportError:
    ALL_VARS          = []   # ⚠️ sera détecté dynamiquement depuis les JSON
    GROUND_TRUTH_DIR  = Path("ground_truth")

# ⚠️  Remplace par tes ~7 champs importants
# Ces champs auront leur propre graphe fill rate + GT accuracy dans le dashboard
IMPORTANT_VARS = [
    "matricula",
    "titular",
    "nif",
    "marca",
    "modelo",
    "bastidor",
    "fecha_matriculacion",
]


# ─── CHARGEMENT ───────────────────────────────────────────────────────────────

def detect_fields_from_json(json_path: Path) -> list[str]:
    """Détecte dynamiquement les champs depuis un JSON si ALL_VARS est vide."""
    with open(json_path, encoding="utf-8") as f:
        data = json.load(f)
    return list(data.get("fields", {}).keys())


def load_all_benchmarks(root: Path, prefix: str) -> dict:
    """
    Parcourt tous les dossiers results_benchmark_*/ et charge les JSON raw.

    Retourne :
    {
      "regex": {
        "paddleocr": {
          "carte_001": {"fields": {...}, "latency_s": 1.2, "error": None},
          ...
        },
        "markitdown": {...}
      },
      "mistral": {...}
    }
    """
    benchmarks = {}
    dirs = sorted([d for d in root.iterdir()
                   if d.is_dir() and d.name.startswith(prefix)])

    if not dirs:
        raise FileNotFoundError(
            f"Aucun dossier '{prefix}*' trouvé dans {root.resolve()}"
        )

    for bench_dir in dirs:
        parser_name = bench_dir.name.replace(prefix, "")  # ex: "regex" ou "mistral"
        raw_dir = bench_dir / "raw"
        if not raw_dir.exists():
            print(f"  ⚠️  Pas de dossier raw/ dans {bench_dir.name} — ignoré")
            continue

        benchmarks[parser_name] = {}
        for model_dir in sorted(raw_dir.iterdir()):
            if not model_dir.is_dir():
                continue
            model_name = model_dir.name
            benchmarks[parser_name][model_name] = {}

            for json_file in sorted(model_dir.glob("*.json")):
                doc_id = json_file.stem
                with open(json_file, encoding="utf-8") as f:
                    data = json.load(f)
                benchmarks[parser_name][model_name][doc_id] = {
                    "fields":    data.get("fields", {}),
                    "latency_s": data.get("latency_s", 0),
                    "error":     data.get("error"),
                }

        print(f"  ✅ {bench_dir.name}  →  "
              f"{len(benchmarks[parser_name])} modèles  |  "
              f"{sum(len(v) for v in benchmarks[parser_name].values())} docs total")

    return benchmarks


def load_ground_truth(gt_dir: Path) -> dict:
    """Retourne {doc_id: {field: value}}."""
    gt = {}
    if not gt_dir.exists():
        return gt
    for p in gt_dir.glob("*.json"):
        with open(p, encoding="utf-8") as f:
            gt[p.stem] = json.load(f)
    return gt


# ─── MÉTRIQUES ────────────────────────────────────────────────────────────────

def cer(ref: str, hyp: str) -> float:
    ref = (ref or "").upper().strip()
    hyp = (hyp or "").upper().strip()
    if not ref:
        return 0.0 if not hyp else 1.0
    m, n = len(ref), len(hyp)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, n + 1):
            cost = 0 if ref[i-1] == hyp[j-1] else 1
            dp[j] = min(dp[j]+1, dp[j-1]+1, prev[j-1]+cost)
    return round(dp[n] / m, 4)


def cross_agreement(docs: dict, all_parsers_models: list[tuple], doc_id: str, field: str) -> float:
    """
    Pour un doc+champ, calcule le % de combinaisons parseur+modèle
    qui sont d'accord avec la valeur majoritaire.
    """
    values = []
    for parser, model, model_docs in all_parsers_models:
        val = model_docs.get(doc_id, {}).get("fields", {}).get(field, "")
        if val:
            values.append(val.upper().strip())
    if not values:
        return 0.0
    majority = max(set(values), key=values.count)
    return sum(1 for v in values if v == majority) / len(values)


# ─── ANALYSE ──────────────────────────────────────────────────────────────────

def analyze(benchmarks: dict, gt: dict, fields: list[str]) -> tuple[list, list]:
    """
    Retourne (global_rows, field_rows) pour les deux CSV.
    """
    # Liste plate de toutes les combinaisons parseur+modèle
    all_combos = [
        (parser, model, model_docs)
        for parser, models in benchmarks.items()
        for model, model_docs in models.items()
    ]

    global_rows = []
    field_rows  = []

    for parser, model, model_docs in all_combos:
        combo_name = f"{parser}+{model}"
        print(f"  Analyse : {combo_name}")

        # Accumulateurs
        latencies        = []
        errors           = 0
        fill_per_field   = defaultdict(list)   # field → [0/1]
        gt_exact         = defaultdict(int)
        gt_cer_vals      = defaultdict(list)
        gt_n             = defaultdict(int)
        cross_agr_vals   = []

        all_doc_ids = list(model_docs.keys())

        for doc_id, doc_data in model_docs.items():
            if doc_data.get("error"):
                errors += 1
                continue

            lat = doc_data.get("latency_s", 0)
            if lat:
                latencies.append(float(lat))

            extracted = doc_data.get("fields", {})

            # Remplissage par champ
            for field in fields:
                val = extracted.get(field, "")
                fill_per_field[field].append(1 if val and str(val).strip() else 0)

            # Ground truth
            if doc_id in gt:
                for field, ref in gt[doc_id].items():
                    if ref is None or field not in fields:
                        continue
                    hyp = extracted.get(field, "")
                    gt_exact[field]    += int((ref or "").upper().strip() ==
                                              (hyp or "").upper().strip())
                    gt_cer_vals[field].append(cer(ref, hyp))
                    gt_n[field]        += 1

            # Cross-validation (accord avec majorité toutes combos)
            field_agreements = []
            for field in fields:
                agr = cross_agreement(model_docs, all_combos, doc_id, field)
                field_agreements.append(agr)
            if field_agreements:
                cross_agr_vals.append(
                    sum(field_agreements) / len(field_agreements)
                )

        # ── Calcul métriques globales ──────────────────────────────────────
        n_docs      = len(all_doc_ids)
        avg_lat     = sum(latencies) / len(latencies) if latencies else 0
        error_rate  = errors / n_docs * 100 if n_docs else 0

        all_fills   = [v for vals in fill_per_field.values() for v in vals]
        fill_rate   = sum(all_fills) / len(all_fills) * 100 if all_fills else 0

        cross_agr   = (sum(cross_agr_vals) / len(cross_agr_vals) * 100
                       if cross_agr_vals else None)

        gt_total_n  = sum(gt_n.values())
        gt_total_ex = sum(gt_exact.values())
        all_cers    = [c for cers in gt_cer_vals.values() for c in cers]
        gt_acc      = gt_total_ex / gt_total_n * 100 if gt_total_n else None
        gt_cer_avg  = sum(all_cers) / len(all_cers) if all_cers else None

        # ── Score global composite ────────────────────────────────────────
        speed_score = max(0, 100 - avg_lat * 10)

        if gt_acc is None and cross_agr is None:
            score = fill_rate * 0.80 + speed_score * 0.20
        elif gt_acc is None:
            score = fill_rate * 0.45 + (cross_agr or 0) * 0.45 + speed_score * 0.10
        elif cross_agr is None:
            score = fill_rate * 0.45 + (gt_acc or 0) * 0.45 + speed_score * 0.10
        else:
            score = (fill_rate * 0.25 + (cross_agr or 0) * 0.25 +
                     (gt_acc or 0) * 0.40 + speed_score * 0.10)

        global_rows.append({
            "combo":                  combo_name,
            "parser":                 parser,
            "ocr_model":              model,
            "n_docs":                 n_docs,
            "n_errors":               errors,
            "error_rate_pct":         round(error_rate, 1),
            "avg_latency_s":          round(avg_lat, 3),
            "fill_rate_pct":          round(fill_rate, 1),
            "cross_agreement_pct":    round(cross_agr, 1) if cross_agr is not None else "N/A",
            "gt_accuracy_pct":        round(gt_acc, 1)    if gt_acc    is not None else "N/A",
            "gt_avg_cer":             round(gt_cer_avg, 4) if gt_cer_avg is not None else "N/A",
            "gt_n_fields_evaluated":  gt_total_n,
            # ── Fill rate par champ important ─────────────────────────────
            **{f"fill_{f}": round(sum(fill_per_field.get(f, []))/len(fill_per_field[f])*100, 1)
               if fill_per_field.get(f) else "N/A"
               for f in IMPORTANT_VARS},
            # ── GT accuracy par champ important ───────────────────────────
            **{f"gt_acc_{f}": round(gt_exact[f]/gt_n[f]*100, 1)
               if gt_n.get(f) else "N/A"
               for f in IMPORTANT_VARS},
            "SCORE_GLOBAL":           round(score, 1),
        })

        # ── Métriques par champ ───────────────────────────────────────────
        for field in fields:
            fills   = fill_per_field.get(field, [])
            fill_f  = sum(fills) / len(fills) * 100 if fills else 0
            gtn     = gt_n[field]
            gt_acc_f  = gt_exact[field] / gtn * 100 if gtn else None
            cers_f    = gt_cer_vals[field]
            cer_f     = sum(cers_f) / len(cers_f) if cers_f else None
            field_rows.append({
                "combo":           combo_name,
                "parser":          parser,
                "ocr_model":       model,
                "field":           field,
                "fill_rate_pct":   round(fill_f, 1),
                "gt_accuracy_pct": round(gt_acc_f, 1) if gt_acc_f is not None else "N/A",
                "gt_avg_cer":      round(cer_f, 4)    if cer_f    is not None else "N/A",
                "gt_n":            gtn,
            })

    # Tri par score décroissant
    global_rows.sort(key=lambda x: float(x["SCORE_GLOBAL"]), reverse=True)
    return global_rows, field_rows


# ─── EXPORT CSV ───────────────────────────────────────────────────────────────

def write_csvs(global_rows: list, field_rows: list, output_dir: Path):
    output_dir.mkdir(parents=True, exist_ok=True)

    global_path = output_dir / "report_global.csv"
    with open(global_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(global_rows[0].keys()))
        writer.writeheader()
        writer.writerows(global_rows)

    field_path = output_dir / "report_by_field.csv"
    with open(field_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(field_rows[0].keys()))
        writer.writeheader()
        writer.writerows(field_rows)

    return global_path, field_path


# ─── DASHBOARD HTML ───────────────────────────────────────────────────────────

# Palette : une couleur par parseur (jusqu'à 8 parseurs)
_PARSER_PALETTE = [
    {"bar": "#378ADD", "border": "#185FA5"},
    {"bar": "#1D9E75", "border": "#0F6E56"},
    {"bar": "#D85A30", "border": "#993C1D"},
    {"bar": "#D4537E", "border": "#993556"},
    {"bar": "#BA7517", "border": "#854F0B"},
    {"bar": "#639922", "border": "#3B6D11"},
    {"bar": "#7F77DD", "border": "#534AB7"},
    {"bar": "#888780", "border": "#5F5E5A"},
]

_METRICS_DEF = [
    ("avg_latency_s",       "Latence moyenne",      "s",    True),
    ("fill_rate_pct",       "Taux de remplissage",  "%",    False),
    ("cross_agreement_pct", "Cross-validation",     "%",    False),
    ("gt_accuracy_pct",     "GT accuracy",          "%",    False),
    ("gt_avg_cer",          "CER moyen",            "",     True),
    ("SCORE_GLOBAL",        "Score global",         "/100", False),
    # Les deux onglets par champ sont gérés séparément (voir generate_html_dashboard)
]


def generate_html_dashboard(global_rows: list, output_dir: Path) -> Path:
    """
    Génère un fichier HTML interactif avec graphes groupés parseur × modèle.
    Un onglet par métrique, barres côte à côte par parseur pour chaque modèle.
    """
    # Déduit parseurs et modèles depuis les données réelles
    parsers = list(dict.fromkeys(r["parser"]    for r in global_rows))
    models  = list(dict.fromkeys(r["ocr_model"] for r in global_rows))

    # Index des données : data[parser][model] = row
    data_index = defaultdict(dict)
    for row in global_rows:
        data_index[row["parser"]][row["ocr_model"]] = row

    # Couleurs par parseur
    parser_colors = {p: _PARSER_PALETTE[i % len(_PARSER_PALETTE)]
                     for i, p in enumerate(parsers)}

    # Synthèse meilleur parseur / modèle
    parser_scores = defaultdict(list)
    model_scores  = defaultdict(list)
    for row in global_rows:
        parser_scores[row["parser"]].append(float(row["SCORE_GLOBAL"]))
        model_scores[row["ocr_model"]].append(float(row["SCORE_GLOBAL"]))
    parser_avg = {p: sum(v)/len(v) for p, v in parser_scores.items()}
    model_avg  = {m: sum(v)/len(v) for m, v in model_scores.items()}
    best_combo  = global_rows[0]
    best_parser = max(parser_avg, key=parser_avg.get)
    best_model  = max(model_avg,  key=model_avg.get)

    # ── Sérialise les données pour JS ─────────────────────────────────────────
    def safe(val):
        try:
            return float(val)
        except (ValueError, TypeError):
            return 0.0

    js_data = {}
    for p in parsers:
        js_data[p] = {}
        for m in models:
            row = data_index[p].get(m, {})
            js_data[p][m] = {k: safe(row.get(k, 0)) for k, *_ in _METRICS_DEF}

    js_parsers       = json.dumps(parsers)
    js_models        = json.dumps(models)
    js_data_str      = json.dumps(js_data)
    js_metrics       = json.dumps([
        {"key": k, "label": lbl, "unit": u, "lowerBetter": lb}
        for k, lbl, u, lb in _METRICS_DEF
    ])
    js_colors        = json.dumps({p: parser_colors[p] for p in parsers})
    js_best_combo    = json.dumps(best_combo["combo"])
    js_best_parser   = json.dumps(best_parser)
    js_best_model    = json.dumps(best_model)
    js_parser_avg    = json.dumps({p: round(v, 1) for p, v in parser_avg.items()})
    js_model_avg     = json.dumps({m: round(v, 1) for m, v in model_avg.items()})
    js_best_score    = json.dumps(float(best_combo["SCORE_GLOBAL"]))
    js_best_p_score  = json.dumps(round(parser_avg[best_parser], 1))
    js_best_m_score  = json.dumps(round(model_avg[best_model], 1))

    # Données par champ important : {parser: {model: {field: value}}}
    important_vars = [v for v in IMPORTANT_VARS if v in (fields or IMPORTANT_VARS)]
    js_important_vars = json.dumps(important_vars)

    fill_by_field = {}   # {parser: {model: {field: value}}}
    gt_by_field   = {}
    for p in parsers:
        fill_by_field[p] = {}
        gt_by_field[p]   = {}
        for m in models:
            row = data_index[p].get(m, {})
            fill_by_field[p][m] = {
                f: safe(row.get(f"fill_{f}", 0)) for f in important_vars
            }
            gt_by_field[p][m] = {
                f: safe(row.get(f"gt_acc_{f}", 0)) for f in important_vars
            }
    js_fill_by_field = json.dumps(fill_by_field)
    js_gt_by_field   = json.dumps(gt_by_field)

    html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Benchmark OCR — Cartes grises ES</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:system-ui,sans-serif;background:#f5f7fa;color:#333;font-size:14px}}
header{{background:#1a2744;color:#fff;padding:20px 32px}}
header h1{{font-size:1.3em;font-weight:500;margin-bottom:4px}}
header p{{opacity:.65;font-size:.85em}}
.container{{max-width:1100px;margin:24px auto;padding:0 20px}}
.winner-row{{display:grid;grid-template-columns:repeat(auto-fit,minmax(180px,1fr));gap:12px;margin-bottom:24px}}
.winner-card{{background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:14px 16px}}
.winner-tag{{font-size:11px;color:#185FA5;background:#E6F1FB;padding:2px 8px;border-radius:6px;display:inline-block;margin-bottom:8px}}
.winner-name{{font-size:15px;font-weight:500}}
.winner-score{{font-size:12px;color:#888;margin-top:2px}}
.tabs{{display:flex;gap:4px;flex-wrap:wrap;margin-bottom:20px}}
.tab{{font-size:12px;padding:5px 14px;border-radius:6px;border:0.5px solid #ccc;background:transparent;color:#666;cursor:pointer}}
.tab.active{{background:#1a2744;color:#fff;border-color:#1a2744;font-weight:500}}
.legend{{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px}}
.legend-item{{display:flex;align-items:center;gap:5px;font-size:12px;color:#666}}
.legend-dot{{width:10px;height:10px;border-radius:2px}}
.chart-wrap{{position:relative;width:100%;background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:16px}}
.synthesis{{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:24px}}
.synth-card{{background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:14px 16px}}
.synth-card h3{{font-size:13px;font-weight:500;color:#555;margin-bottom:10px}}
.synth-row{{display:flex;justify-content:space-between;align-items:center;padding:4px 0;border-bottom:0.5px solid #f0f0f0;font-size:13px}}
.synth-row:last-child{{border-bottom:none}}
.synth-val{{font-weight:500;color:#1a2744}}
.note{{font-size:11px;color:#999;margin-top:16px;line-height:1.6}}
</style>
</head>
<body>
<header>
  <h1>Benchmark OCR — Cartes grises espagnoles</h1>
  <p>Comparaison parseurs × modèles OCR</p>
</header>
<div class="container">

  <div class="winner-row" id="winnerRow"></div>
  <div class="tabs" id="tabBar"></div>
  <div class="legend" id="legend"></div>
  <div class="chart-wrap"><div id="chartWrap" style="position:relative"></div></div>

  <div class="synthesis">
    <div class="synth-card">
      <h3>Score moyen par parseur</h3>
      <div id="parserSynth"></div>
    </div>
    <div class="synth-card">
      <h3>Score moyen par modèle OCR</h3>
      <div id="modelSynth"></div>
    </div>
  </div>

  <p class="note">
    Score global = taux de remplissage (25%) + accord cross-validation (25%) + GT accuracy (40%) + rapidité (10%).
    Les métriques N/A sont exclues et les poids redistribués. Latence et CER : plus bas = mieux.
  </p>
</div>

<script>
const PARSERS    = {js_parsers};
const MODELS     = {js_models};
const DATA       = {js_data_str};
const METRICS    = {js_metrics};
const COLORS     = {js_colors};
const BEST_COMBO  = {js_best_combo};
const BEST_PARSER = {js_best_parser};
const BEST_MODEL  = {js_best_model};
const PARSER_AVG  = {js_parser_avg};
const MODEL_AVG   = {js_model_avg};
const BEST_SCORE  = {js_best_score};
const BEST_P_SCORE = {js_best_p_score};
const BEST_M_SCORE = {js_best_m_score};
const IMPORTANT_VARS  = {js_important_vars};
const FILL_BY_FIELD   = {js_fill_by_field};
const GT_BY_FIELD     = {js_gt_by_field};

// Onglets spéciaux par champ
const FIELD_TABS = [
  {{ key: 'fill_by_field', label: 'Fill rate par champ',    data: FILL_BY_FIELD, unit: '%' }},
  {{ key: 'gt_by_field',   label: 'GT accuracy par champ',  data: GT_BY_FIELD,   unit: '%' }},
];

let activeMetric = METRICS[5].key;  // Score global par défaut
let activeFieldTab = null;
let currentChart = null;

function buildWinners() {{
  document.getElementById('winnerRow').innerHTML = `
    <div class="winner-card"><span class="winner-tag">Meilleure combo</span>
      <div class="winner-name">${{BEST_COMBO}}</div>
      <div class="winner-score">${{BEST_SCORE.toFixed(1)}} / 100</div></div>
    <div class="winner-card"><span class="winner-tag">Meilleur parseur</span>
      <div class="winner-name">${{BEST_PARSER}}</div>
      <div class="winner-score">moy. ${{BEST_P_SCORE.toFixed(1)}} / 100</div></div>
    <div class="winner-card"><span class="winner-tag">Meilleur modèle OCR</span>
      <div class="winner-name">${{BEST_MODEL}}</div>
      <div class="winner-score">moy. ${{BEST_M_SCORE.toFixed(1)}} / 100</div></div>`;
}}

function buildLegend() {{
  document.getElementById('legend').innerHTML = PARSERS.map(p =>
    `<span class="legend-item">
      <span class="legend-dot" style="background:${{COLORS[p].bar}}"></span>${{p}}
    </span>`).join('');
}}

function buildTabs() {{
  const bar = document.getElementById('tabBar');

  // Onglets métriques globales
  METRICS.forEach(m => {{
    const btn = document.createElement('button');
    btn.className = 'tab' + (m.key === activeMetric ? ' active' : '');
    btn.textContent = m.label;
    btn.dataset.key = m.key;
    btn.onclick = () => {{
      activeMetric  = m.key;
      activeFieldTab = null;
      renderChart();
      updateTabs();
    }};
    bar.appendChild(btn);
  }});

  // Séparateur visuel
  const sep = document.createElement('span');
  sep.style.cssText = 'width:1px;background:#ddd;margin:0 4px;align-self:stretch';
  bar.appendChild(sep);

  // Onglets par champ important
  FIELD_TABS.forEach(ft => {{
    const btn = document.createElement('button');
    btn.className = 'tab';
    btn.textContent = ft.label;
    btn.dataset.key = ft.key;
    btn.onclick = () => {{
      activeFieldTab = ft.key;
      activeMetric   = null;
      renderChart();
      updateTabs();
    }};
    bar.appendChild(btn);
  }});
}}

function updateTabs() {{
  document.querySelectorAll('.tab').forEach(t => {{
    t.classList.toggle('active',
      t.dataset.key === (activeFieldTab || activeMetric));
  }});
}}

function renderChart() {{
  const wrap = document.getElementById('chartWrap');
  if (currentChart) {{ currentChart.destroy(); currentChart = null; }}

  // ── Onglet par champ ────────────────────────────────────────────────────────
  if (activeFieldTab) {{
    const ft     = FIELD_TABS.find(f => f.key === activeFieldTab);
    const labels = IMPORTANT_VARS;
    // Pour chaque combo parseur+modèle, une série
    const datasets = [];
    const palette  = ['#378ADD','#1D9E75','#D85A30','#D4537E','#BA7517','#639922','#7F77DD','#888780'];
    let ci = 0;
    PARSERS.forEach(p => {{
      MODELS.forEach(m => {{
        const color = palette[ci % palette.length];
        datasets.push({{
          label: `${{p}}+${{m}}`,
          data:  labels.map(f => ft.data[p]?.[m]?.[f] ?? 0),
          backgroundColor: color + 'BB',
          borderColor:     color,
          borderWidth: 1,
          borderRadius: 3,
        }});
        ci++;
      }});
    }});

    const h = Math.max(labels.length * (datasets.length * 22 + 20) + 80, 300);
    wrap.style.height = h + 'px';
    wrap.innerHTML = `<canvas id="mainChart" role="img" aria-label="${{ft.label}} par champ" style="display:block"></canvas>`;

    currentChart = new Chart(document.getElementById('mainChart'), {{
      type: 'bar',
      data: {{ labels, datasets }},
      options: {{
        indexAxis: 'y',
        responsive: true,
        maintainAspectRatio: false,
        plugins: {{
          legend: {{
            display: true,
            position: 'top',
            labels: {{ font: {{ size: 11 }}, boxWidth: 12, padding: 10 }}
          }},
          tooltip: {{
            callbacks: {{
              label: ctx => ` ${{ctx.dataset.label}}: ${{ctx.parsed.x.toFixed(1)}}${{ft.unit}}`
            }}
          }}
        }},
        scales: {{
          x: {{
            beginAtZero: true, max: 100,
            grid: {{ color: 'rgba(0,0,0,0.05)' }},
            ticks: {{ font: {{ size: 11 }}, color: '#888', callback: v => v + ft.unit }}
          }},
          y: {{ ticks: {{ font: {{ size: 12 }}, color: '#555' }}, grid: {{ display: false }} }}
        }}
      }}
    }});
    return;
  }}

  // ── Onglet métrique globale ─────────────────────────────────────────────────
  const metric = METRICS.find(m => m.key === activeMetric);
  const h      = Math.max(MODELS.length * 72 + 60, 200);
  wrap.style.height = h + 'px';
  wrap.innerHTML = `<canvas id="mainChart" role="img"
    aria-label="Graphe ${{metric.label}} par modèle et parseur"
    style="display:block"></canvas>`;

  const datasets = PARSERS.map(p => ({{
    label: p,
    data:  MODELS.map(m => DATA[p]?.[m]?.[activeMetric] ?? 0),
    backgroundColor: COLORS[p].bar + 'CC',
    borderColor:     COLORS[p].border,
    borderWidth: 1,
    borderRadius: 3,
  }}));

  currentChart = new Chart(document.getElementById('mainChart'), {{
    type: 'bar',
    data: {{ labels: MODELS, datasets }},
    options: {{
      indexAxis: 'y',
      responsive: true,
      maintainAspectRatio: false,
      plugins: {{
        legend: {{ display: false }},
        tooltip: {{
          callbacks: {{
            label: ctx => ` ${{ctx.dataset.label}}: ${{ctx.parsed.x.toFixed(
              metric.key === 'gt_avg_cer' ? 3 : 1)}}${{metric.unit}}`
          }}
        }}
      }},
      scales: {{
        x: {{
          beginAtZero: true,
          grid: {{ color: 'rgba(0,0,0,0.05)' }},
          ticks: {{
            font: {{ size: 11 }}, color: '#888',
            callback: v => v + metric.unit
          }}
        }},
        y: {{
          ticks: {{ font: {{ size: 12 }}, color: '#555' }},
          grid: {{ display: false }}
        }}
      }}
    }}
  }});
}}

function buildSynthesis() {{
  document.getElementById('parserSynth').innerHTML =
    Object.entries(PARSER_AVG).sort((a,b) => b[1]-a[1]).map(([p,v]) =>
      `<div class="synth-row">
        <span style="display:flex;align-items:center;gap:6px">
          <span style="width:8px;height:8px;border-radius:2px;background:${{COLORS[p].bar}};display:inline-block"></span>
          ${{p}}
        </span>
        <span class="synth-val">${{v.toFixed(1)}} / 100</span>
      </div>`).join('');

  document.getElementById('modelSynth').innerHTML =
    Object.entries(MODEL_AVG).sort((a,b) => b[1]-a[1]).map(([m,v]) =>
      `<div class="synth-row"><span>${{m}}</span>
        <span class="synth-val">${{v.toFixed(1)}} / 100</span>
      </div>`).join('');
}}

buildWinners();
buildLegend();
buildTabs();
renderChart();
buildSynthesis();
</script>
</body>
</html>"""

    html_path = output_dir / "dashboard.html"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    return html_path


# ─── AFFICHAGE TERMINAL ───────────────────────────────────────────────────────

def print_ranking(global_rows: list):
    print("\n" + "═" * 70)
    print("  CLASSEMENT FINAL — PARSEURS × MODÈLES OCR")
    print("═" * 70)

    icons = ["🥇", "🥈", "🥉"] + ["  "] * 20
    for i, row in enumerate(global_rows):
        print(f"\n  {icons[i]}  {row['combo'].upper()}")
        print(f"       Score global      : {row['SCORE_GLOBAL']:>6} / 100")
        print(f"       Latence moyenne   : {row['avg_latency_s']:>6}s")
        print(f"       Taux remplissage  : {row['fill_rate_pct']:>6}%")
        print(f"       Cross-validation  : {str(row['cross_agreement_pct']):>6}%")
        print(f"       GT accuracy       : {str(row['gt_accuracy_pct']):>6}%  "
              f"(CER {row['gt_avg_cer']})")
        print(f"       Erreurs           : {row['n_errors']}/{row['n_docs']} docs")

    parser_scores = defaultdict(list)
    model_scores  = defaultdict(list)
    for row in global_rows:
        parser_scores[row["parser"]].append(float(row["SCORE_GLOBAL"]))
        model_scores[row["ocr_model"]].append(float(row["SCORE_GLOBAL"]))
    parser_avg = {p: sum(v)/len(v) for p, v in parser_scores.items()}
    model_avg  = {m: sum(v)/len(v) for m, v in model_scores.items()}

    print("\n" + "─" * 70)
    print("  SYNTHÈSE")
    print("─" * 70)
    print(f"\n  Meilleur parseur  : {max(parser_avg, key=parser_avg.get).upper()}")
    for p, avg in sorted(parser_avg.items(), key=lambda x: -x[1]):
        print(f"    {p:<20} {avg:.1f} / 100")
    print(f"\n  Meilleur modèle OCR : {max(model_avg, key=model_avg.get).upper()}")
    for m, avg in sorted(model_avg.items(), key=lambda x: -x[1]):
        print(f"    {m:<20} {avg:.1f} / 100")
    print("\n  Score = remplissage 25% + cross-valid. 25% + GT acc. 40% + rapidité 10%\n")


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--root",   type=Path, default=DEFAULT_ROOT,
                        help="Dossier parent contenant les results_benchmark_*/")
    parser.add_argument("--prefix", type=str,  default=BENCHMARK_PREFIX,
                        help="Préfixe des dossiers benchmark")
    parser.add_argument("--gt",     type=Path, default=GROUND_TRUTH_DIR,
                        help="Dossier ground truth")
    parser.add_argument("--output", type=Path, default=OUTPUT_DIR,
                        help="Dossier de sortie")
    args = parser.parse_args()

    print(f"\n🔍 Recherche des benchmarks dans : {args.root.resolve()}")
    print(f"   Préfixe : {args.prefix}\n")

    benchmarks = load_all_benchmarks(args.root, args.prefix)
    gt         = load_ground_truth(args.gt)
    print(f"\n   Ground truth : {len(gt)} docs annotés")

    fields = ALL_VARS
    if not fields:
        for models in benchmarks.values():
            for model_docs in models.values():
                for doc_data in model_docs.values():
                    detected = list(doc_data["fields"].keys())
                    if detected:
                        fields = detected
                        break
                if fields:
                    break
            if fields:
                break
        print(f"   Champs détectés automatiquement : {fields}")

    if not fields:
        print("❌ Impossible de détecter les champs. Vérifie tes JSON raw.")
        return

    print("\n📊 Calcul des métriques...\n")
    global_rows, field_rows = analyze(benchmarks, gt, fields)

    # CSV
    global_path, field_path = write_csvs(global_rows, field_rows, args.output)

    # Dashboard HTML
    html_path = generate_html_dashboard(global_rows, args.output)

    # Classement terminal
    print_ranking(global_rows)

    print(f"  💾  report_global.csv   → {global_path}")
    print(f"  💾  report_by_field.csv → {field_path}")
    print(f"  🌐  dashboard.html      → {html_path}\n")


if __name__ == "__main__":
    main()

In [ ]:
"""
Phase 4 — Rapport final.
Agrège latence, fill rate, GT accuracy et génère :
  - Un dashboard HTML interactif avec graphes groupés
  - Un CSV récapitulatif

Métriques :
  - Latence moyenne par modèle
  - Taux de remplissage global
  - Taux de remplissage par champ important (IMPORTANT_VARS)
  - GT accuracy globale
  - GT accuracy par champ important (IMPORTANT_VARS)

Usage :
    python phase4_report.py
"""

import json
import csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict

from config import (
    RESULTS_RAW_DIR, RESULTS_GT_DIR,
    RESULTS_REPORT, ACTIVE_MODELS, ALL_VARS
)

# ⚠️  Remplace par tes ~7 champs les plus importants
IMPORTANT_VARS = [
    "matricula",
    "titular",
    "nif",
    "marca",
    "modelo",
    "bastidor",
    "fecha_matriculacion",
]

# Palette de couleurs par modèle
_PALETTE = [
    {"bar": "#4e79a7", "border": "#2e5980"},
    {"bar": "#f28e2b", "border": "#c06a10"},
    {"bar": "#e15759", "border": "#b03335"},
    {"bar": "#76b7b2", "border": "#4a9490"},
    {"bar": "#59a14f", "border": "#377a2e"},
    {"bar": "#D4537E", "border": "#993556"},
    {"bar": "#BA7517", "border": "#854F0B"},
    {"bar": "#7F77DD", "border": "#534AB7"},
]


# ─── CHARGEMENT ───────────────────────────────────────────────────────────────

def load_latencies() -> dict:
    """Latence moyenne par modèle."""
    latencies = defaultdict(list)
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if not model_dir.exists():
            continue
        for p in model_dir.glob("*.json"):
            with open(p) as f:
                data = json.load(f)
            if data.get("latency_s"):
                latencies[model].append(float(data["latency_s"]))
    return {m: (sum(v)/len(v) if v else 0) for m, v in latencies.items()}


def load_fill_rates() -> dict:
    """
    Retourne :
      global_fill  : {model: pct}
      field_fill   : {model: {field: pct}}  pour IMPORTANT_VARS seulement
    """
    filled_global  = defaultdict(int)
    total_global   = defaultdict(int)
    filled_field   = defaultdict(lambda: defaultdict(int))
    total_field    = defaultdict(lambda: defaultdict(int))

    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if not model_dir.exists():
            continue
        for p in model_dir.glob("*.json"):
            with open(p) as f:
                data = json.load(f)
            if data.get("error"):
                continue
            fields = data.get("fields", {})
            for field in ALL_VARS:
                total_global[model] += 1
                if fields.get(field):
                    filled_global[model] += 1
            for field in IMPORTANT_VARS:
                total_field[model][field] += 1
                if fields.get(field):
                    filled_field[model][field] += 1

    global_fill = {
        m: (filled_global[m] / total_global[m] * 100 if total_global[m] else 0)
        for m in ACTIVE_MODELS
    }
    field_fill = {
        m: {
            f: (filled_field[m][f] / total_field[m][f] * 100
                if total_field[m].get(f) else 0)
            for f in IMPORTANT_VARS
        }
        for m in ACTIVE_MODELS
    }
    return global_fill, field_fill


def load_gt_scores() -> dict:
    """
    Retourne :
      global_gt   : {model: accuracy_pct}
      field_gt    : {model: {field: accuracy_pct}}  pour IMPORTANT_VARS seulement
    """
    gt_files = sorted(RESULTS_GT_DIR.glob("gt_eval_summary_*.json"))
    if not gt_files:
        return {}, {}

    with open(gt_files[-1]) as f:
        rows = json.load(f)

    # global
    model_acc = defaultdict(list)
    for row in rows:
        model_acc[row["model"]].append(row["accuracy_pct"])
    global_gt = {m: (sum(v)/len(v) if v else 0) for m, v in model_acc.items()}

    # par champ important
    field_acc = defaultdict(lambda: defaultdict(list))
    for row in rows:
        if row.get("field") in IMPORTANT_VARS:
            field_acc[row["model"]][row["field"]].append(row["accuracy_pct"])
    field_gt = {
        m: {
            f: (sum(field_acc[m][f])/len(field_acc[m][f])
                if field_acc[m].get(f) else None)
            for f in IMPORTANT_VARS
        }
        for m in ACTIVE_MODELS
    }
    return global_gt, field_gt


# ─── CSV ──────────────────────────────────────────────────────────────────────

def write_csv(ts: str, latencies: dict, global_fill: dict,
              field_fill: dict, global_gt: dict, field_gt: dict) -> Path:
    """
    Génère le CSV récapitulatif.

    Colonnes :
      model | avg_latency_s | fill_rate_pct | gt_accuracy_pct
      + fill_<champ>    pour chaque IMPORTANT_VARS
      + gt_acc_<champ>  pour chaque IMPORTANT_VARS
    """
    fieldnames = (
        ["model", "avg_latency_s", "fill_rate_pct", "gt_accuracy_pct"]
        + [f"fill_{f}"   for f in IMPORTANT_VARS]
        + [f"gt_acc_{f}" for f in IMPORTANT_VARS]
    )

    csv_path = RESULTS_REPORT / f"summary_{ts}.csv"
    RESULTS_REPORT.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for model in ACTIVE_MODELS:
            row = {
                "model":           model,
                "avg_latency_s":   round(latencies.get(model, 0), 3),
                "fill_rate_pct":   round(global_fill.get(model, 0), 1),
                "gt_accuracy_pct": round(global_gt.get(model, 0), 1)
                                   if global_gt.get(model) is not None else "N/A",
            }
            for f in IMPORTANT_VARS:
                row[f"fill_{f}"]   = round(field_fill.get(model, {}).get(f, 0), 1)
                gt_val = field_gt.get(model, {}).get(f)
                row[f"gt_acc_{f}"] = round(gt_val, 1) if gt_val is not None else "N/A"
            writer.writerow(row)

    return csv_path


# ─── HTML DASHBOARD ───────────────────────────────────────────────────────────

def generate_html(ts: str, latencies: dict, global_fill: dict, field_fill: dict,
                  global_gt: dict, field_gt: dict) -> Path:

    models = ACTIVE_MODELS
    colors = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(models)}

    def safe(v):
        try: return round(float(v), 1)
        except: return 0.0

    # Données JS
    import json as _json
    js_models    = _json.dumps(models)
    js_colors    = _json.dumps(colors)
    js_imp_vars  = _json.dumps(IMPORTANT_VARS)
    js_latencies = _json.dumps({m: safe(latencies.get(m, 0)) for m in models})
    js_fill      = _json.dumps({m: safe(global_fill.get(m, 0)) for m in models})
    js_gt        = _json.dumps({m: safe(global_gt.get(m, 0)) for m in models})
    js_fill_f    = _json.dumps({
        m: {f: safe(field_fill.get(m, {}).get(f, 0)) for f in IMPORTANT_VARS}
        for m in models
    })
    js_gt_f = _json.dumps({
        m: {f: safe(field_gt.get(m, {}).get(f) or 0) for f in IMPORTANT_VARS}
        for m in models
    })

    html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Benchmark OCR — Cartes grises ES</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:system-ui,sans-serif;background:#f5f7fa;color:#333;font-size:14px}}
header{{background:#1a2744;color:#fff;padding:20px 32px}}
header h1{{font-size:1.25em;font-weight:500;margin-bottom:3px}}
header p{{opacity:.6;font-size:.82em}}
.container{{max-width:1100px;margin:24px auto;padding:0 20px}}
.chips{{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:20px}}
.chip{{padding:5px 14px;border-radius:20px;color:#fff;font-size:.82em;font-weight:500}}
.tabs{{display:flex;gap:4px;flex-wrap:wrap;margin-bottom:16px}}
.tab{{font-size:12px;padding:5px 14px;border-radius:6px;border:0.5px solid #ccc;
      background:transparent;color:#666;cursor:pointer}}
.tab.active{{background:#1a2744;color:#fff;border-color:#1a2744;font-weight:500}}
.sep{{width:1px;background:#ddd;margin:0 4px;align-self:stretch}}
.legend{{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px}}
.legend-item{{display:flex;align-items:center;gap:5px;font-size:12px;color:#666}}
.legend-dot{{width:10px;height:10px;border-radius:2px}}
.chart-card{{background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:16px;margin-bottom:20px}}
.note{{font-size:11px;color:#999;margin-top:16px;line-height:1.6}}
</style>
</head>
<body>
<header>
  <h1>Benchmark OCR — Cartes grises espagnoles</h1>
  <p>Généré le {datetime.now().strftime("%d/%m/%Y %H:%M")} · {len(models)} modèles · {len(ALL_VARS)} champs</p>
</header>
<div class="container">
  <div class="chips" id="chips"></div>
  <div class="tabs" id="tabBar"></div>
  <div class="legend" id="legend"></div>
  <div class="chart-card"><div id="chartWrap" style="position:relative"></div></div>
  <p class="note">
    Fill rate = % de champs non vides extraits sur l'ensemble des docs.<br>
    GT accuracy = % de champs exactement corrects vs ground truth (N/A si aucun doc annoté).<br>
    Latence : plus bas = mieux. Fill rate et GT accuracy : plus haut = mieux.
  </p>
</div>
<script>
const MODELS       = {js_models};
const COLORS       = {js_colors};
const IMP_VARS     = {js_imp_vars};
const LATENCIES    = {js_latencies};
const FILL         = {js_fill};
const GT           = {js_gt};
const FILL_FIELD   = {js_fill_f};
const GT_FIELD     = {js_gt_f};

// Onglets globaux
const GLOBAL_TABS = [
  {{ key:'latency',  label:'Latence',          data:LATENCIES, unit:'s',   axis:'x', lowerBetter:true  }},
  {{ key:'fill',     label:'Fill rate global', data:FILL,      unit:'%',   axis:'x', lowerBetter:false }},
  {{ key:'gt',       label:'GT accuracy',      data:GT,        unit:'%',   axis:'x', lowerBetter:false }},
];
// Onglets par champ important
const FIELD_TABS = [
  {{ key:'fill_field', label:'Fill rate par champ',   data:FILL_FIELD, unit:'%' }},
  {{ key:'gt_field',   label:'GT accuracy par champ', data:GT_FIELD,   unit:'%' }},
];

let active = 'fill';
let chart  = null;

// Chips modèles
document.getElementById('chips').innerHTML = MODELS.map(m =>
  `<span class="chip" style="background:${{COLORS[m].bar}}">${{m}}</span>`).join('');

// Légende
document.getElementById('legend').innerHTML = MODELS.map(m =>
  `<span class="legend-item">
    <span class="legend-dot" style="background:${{COLORS[m].bar}}"></span>${{m}}
  </span>`).join('');

// Onglets
const bar = document.getElementById('tabBar');
GLOBAL_TABS.forEach(t => {{
  const btn = document.createElement('button');
  btn.className = 'tab' + (t.key === active ? ' active' : '');
  btn.textContent = t.label; btn.dataset.key = t.key;
  btn.onclick = () => {{ active = t.key; render(); updateTabs(); }};
  bar.appendChild(btn);
}});
const sep = document.createElement('span');
sep.className = 'sep'; bar.appendChild(sep);
FIELD_TABS.forEach(t => {{
  const btn = document.createElement('button');
  btn.className = 'tab'; btn.textContent = t.label; btn.dataset.key = t.key;
  btn.onclick = () => {{ active = t.key; render(); updateTabs(); }};
  bar.appendChild(btn);
}});

function updateTabs() {{
  document.querySelectorAll('.tab').forEach(t =>
    t.classList.toggle('active', t.dataset.key === active));
}}

function render() {{
  const wrap = document.getElementById('chartWrap');
  if (chart) {{ chart.destroy(); chart = null; }}

  // ── Onglet par champ ──────────────────────────────────────────────────────
  const ft = FIELD_TABS.find(t => t.key === active);
  if (ft) {{
    const h = Math.max(IMP_VARS.length * (MODELS.length * 22 + 16) + 80, 300);
    wrap.style.height = h + 'px';
    wrap.innerHTML = `<canvas id="mc" role="img" aria-label="${{ft.label}} par champ" style="display:block"></canvas>`;
    chart = new Chart(document.getElementById('mc'), {{
      type: 'bar',
      data: {{
        labels: IMP_VARS,
        datasets: MODELS.map(m => ({{
          label: m,
          data: IMP_VARS.map(f => ft.data[m]?.[f] ?? 0),
          backgroundColor: COLORS[m].bar + 'CC',
          borderColor: COLORS[m].border,
          borderWidth: 1, borderRadius: 3,
        }}))
      }},
      options: {{
        indexAxis: 'y', responsive: true, maintainAspectRatio: false,
        plugins: {{
          legend: {{ display: true, position: 'top',
            labels: {{ font: {{ size: 11 }}, boxWidth: 12, padding: 10 }} }},
          tooltip: {{ callbacks: {{ label: ctx =>
            ` ${{ctx.dataset.label}}: ${{ctx.parsed.x.toFixed(1)}}${{ft.unit}}` }} }}
        }},
        scales: {{
          x: {{ beginAtZero:true, max:100,
            grid: {{ color:'rgba(0,0,0,0.05)' }},
            ticks: {{ font:{{size:11}}, color:'#888', callback: v => v+ft.unit }} }},
          y: {{ ticks: {{ font:{{size:12}}, color:'#555' }}, grid:{{display:false}} }}
        }}
      }}
    }});
    return;
  }}

  // ── Onglet global ─────────────────────────────────────────────────────────
  const gt = GLOBAL_TABS.find(t => t.key === active);
  const h  = Math.max(MODELS.length * 52 + 60, 160);
  wrap.style.height = h + 'px';
  wrap.innerHTML = `<canvas id="mc" role="img" aria-label="${{gt.label}}" style="display:block"></canvas>`;
  chart = new Chart(document.getElementById('mc'), {{
    type: 'bar',
    data: {{
      labels: MODELS,
      datasets: [{{
        data: MODELS.map(m => gt.data[m] ?? 0),
        backgroundColor: MODELS.map(m => COLORS[m].bar + 'CC'),
        borderColor:     MODELS.map(m => COLORS[m].border),
        borderWidth: 1, borderRadius: 4,
      }}]
    }},
    options: {{
      indexAxis: 'y', responsive: true, maintainAspectRatio: false,
      plugins: {{
        legend: {{ display: false }},
        tooltip: {{ callbacks: {{ label: ctx =>
          ` ${{ctx.parsed.x.toFixed(gt.key === 'latency' ? 2 : 1)}}${{gt.unit}}` }} }}
      }},
      scales: {{
        x: {{ beginAtZero:true,
          grid: {{ color:'rgba(0,0,0,0.05)' }},
          ticks: {{ font:{{size:11}}, color:'#888', callback: v => v+gt.unit }} }},
        y: {{ ticks: {{ font:{{size:12}}, color:'#555' }}, grid:{{display:false}} }}
      }}
    }}
  }});
}}

render();
</script>
</body>
</html>"""

    html_path = RESULTS_REPORT / f"report_{ts}.html"
    RESULTS_REPORT.mkdir(parents=True, exist_ok=True)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    return html_path


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    print("📊 Génération du rapport final...\n")

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    latencies             = load_latencies()
    global_fill, field_fill = load_fill_rates()
    global_gt,   field_gt   = load_gt_scores()

    # CSV
    csv_path  = write_csv(ts, latencies, global_fill, field_fill, global_gt, field_gt)

    # HTML
    html_path = generate_html(ts, latencies, global_fill, field_fill, global_gt, field_gt)

    # Résumé terminal
    print(f"  {'MODÈLE':<15} {'LATENCE':>8} {'FILL':>8} {'GT ACC':>8}")
    print("  " + "─" * 45)
    for model in ACTIVE_MODELS:
        gt_val = global_gt.get(model)
        gt_str = f"{gt_val:.1f}%" if gt_val else "N/A"
        print(f"  {model:<15} {latencies.get(model, 0):>7.2f}s "
              f"{global_fill.get(model, 0):>7.1f}% {gt_str:>8}")

    print(f"\n  💾  CSV     → {csv_path}")
    print(f"  🌐  HTML    → {html_path}\n")


if __name__ == "__main__":
    run()

In [ ]:
"""
Phase 4 — Rapport final.
Agrège latence, fill rate, GT accuracy et génère :
  - Un dashboard HTML interactif avec graphes groupés
  - Un CSV récapitulatif

Métriques :
  - Latence moyenne par modèle
  - Taux de remplissage global
  - Taux de remplissage par champ important (IMPORTANT_VARS)
  - GT accuracy globale
  - GT accuracy par champ important (IMPORTANT_VARS)

Usage :
    python phase4_report.py
"""

import json
import csv
from pathlib import Path
from datetime import datetime
from collections import defaultdict

from config import (
    RESULTS_RAW_DIR, RESULTS_GT_DIR,
    RESULTS_REPORT, ACTIVE_MODELS, ALL_VARS
)

# ⚠️  Remplace par tes ~7 champs les plus importants
IMPORTANT_VARS = [
    "matricula",
    "titular",
    "nif",
    "marca",
    "modelo",
    "bastidor",
    "fecha_matriculacion",
]

# Palette de couleurs par modèle
_PALETTE = [
    {"bar": "#4e79a7", "border": "#2e5980"},
    {"bar": "#f28e2b", "border": "#c06a10"},
    {"bar": "#e15759", "border": "#b03335"},
    {"bar": "#76b7b2", "border": "#4a9490"},
    {"bar": "#59a14f", "border": "#377a2e"},
    {"bar": "#D4537E", "border": "#993556"},
    {"bar": "#BA7517", "border": "#854F0B"},
    {"bar": "#7F77DD", "border": "#534AB7"},
]


# ─── CHARGEMENT ───────────────────────────────────────────────────────────────

def load_latencies() -> dict:
    """Latence moyenne par modèle."""
    latencies = defaultdict(list)
    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if not model_dir.exists():
            continue
        for p in model_dir.glob("*.json"):
            with open(p) as f:
                data = json.load(f)
            if data.get("latency_s"):
                latencies[model].append(float(data["latency_s"]))
    return {m: (sum(v)/len(v) if v else 0) for m, v in latencies.items()}


def load_fill_rates() -> dict:
    """
    Retourne :
      global_fill  : {model: pct}
      field_fill   : {model: {field: pct}}  pour IMPORTANT_VARS seulement
    """
    filled_global  = defaultdict(int)
    total_global   = defaultdict(int)
    filled_field   = defaultdict(lambda: defaultdict(int))
    total_field    = defaultdict(lambda: defaultdict(int))

    for model in ACTIVE_MODELS:
        model_dir = RESULTS_RAW_DIR / model
        if not model_dir.exists():
            continue
        for p in model_dir.glob("*.json"):
            with open(p) as f:
                data = json.load(f)
            if data.get("error"):
                continue
            fields = data.get("fields", {})
            for field in ALL_VARS:
                total_global[model] += 1
                if fields.get(field):
                    filled_global[model] += 1
            for field in IMPORTANT_VARS:
                total_field[model][field] += 1
                if fields.get(field):
                    filled_field[model][field] += 1

    global_fill = {
        m: (filled_global[m] / total_global[m] * 100 if total_global[m] else 0)
        for m in ACTIVE_MODELS
    }
    field_fill = {
        m: {
            f: (filled_field[m][f] / total_field[m][f] * 100
                if total_field[m].get(f) else 0)
            for f in IMPORTANT_VARS
        }
        for m in ACTIVE_MODELS
    }
    return global_fill, field_fill


def load_gt_scores() -> tuple[dict, dict]:
    """
    Lit le dernier gt_eval_detail_*.csv (généré par phase3_gt_eval.py)
    et agrège sur la colonne exact_match (True/False).

    Règles appliquées par phase3 (déjà dans le CSV) :
      - GT = 'x'  et extraction = 'x'  → True
      - GT = 'x'  et extraction = ''   → False
      - GT = ''   et extraction = ''   → True
      - GT absent                       → ligne absente du CSV (ignorée)

    Retourne :
      global_gt : {model: accuracy_pct}
      field_gt  : {model: {field: accuracy_pct}}  pour IMPORTANT_VARS
    """
    detail_files = sorted(RESULTS_GT_DIR.glob("gt_eval_detail_*.csv"))
    if not detail_files:
        return {}, {}

    # Prend le fichier le plus récent
    latest = detail_files[-1]

    global_exact = defaultdict(int)
    global_total = defaultdict(int)
    field_exact  = defaultdict(lambda: defaultdict(int))
    field_total  = defaultdict(lambda: defaultdict(int))

    with open(latest, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            model = row.get("model")
            field = row.get("field")
            exact = row.get("exact_match", "").strip().lower()

            if not model or not field:
                continue

            is_correct = exact in ("true", "1", "yes")

            global_exact[model] += int(is_correct)
            global_total[model] += 1

            if field in IMPORTANT_VARS:
                field_exact[model][field] += int(is_correct)
                field_total[model][field] += 1

    global_gt = {
        m: (global_exact[m] / global_total[m] * 100 if global_total[m] else None)
        for m in ACTIVE_MODELS
    }
    field_gt = {
        m: {
            f: (field_exact[m][f] / field_total[m][f] * 100
                if field_total[m].get(f) else None)
            for f in IMPORTANT_VARS
        }
        for m in ACTIVE_MODELS
    }
    return global_gt, field_gt


# ─── CSV ──────────────────────────────────────────────────────────────────────

def write_csv(ts: str, latencies: dict, global_fill: dict,
              field_fill: dict, global_gt: dict, field_gt: dict) -> Path:
    """
    Génère le CSV récapitulatif.

    Colonnes :
      model | avg_latency_s | fill_rate_pct | gt_accuracy_pct
      + fill_<champ>    pour chaque IMPORTANT_VARS
      + gt_acc_<champ>  pour chaque IMPORTANT_VARS
    """
    fieldnames = (
        ["model", "avg_latency_s", "fill_rate_pct", "gt_accuracy_pct"]
        + [f"fill_{f}"   for f in IMPORTANT_VARS]
        + [f"gt_acc_{f}" for f in IMPORTANT_VARS]
    )

    csv_path = RESULTS_REPORT / f"summary_{ts}.csv"
    RESULTS_REPORT.mkdir(parents=True, exist_ok=True)
    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for model in ACTIVE_MODELS:
            row = {
                "model":           model,
                "avg_latency_s":   round(latencies.get(model, 0), 3),
                "fill_rate_pct":   round(global_fill.get(model, 0), 1),
                "gt_accuracy_pct": round(global_gt.get(model, 0), 1)
                                   if global_gt.get(model) is not None else "N/A",
            }
            for f in IMPORTANT_VARS:
                row[f"fill_{f}"]   = round(field_fill.get(model, {}).get(f, 0), 1)
                gt_val = field_gt.get(model, {}).get(f)
                row[f"gt_acc_{f}"] = round(gt_val, 1) if gt_val is not None else "N/A"
            writer.writerow(row)

    return csv_path


# ─── HTML DASHBOARD ───────────────────────────────────────────────────────────

def generate_html(ts: str, latencies: dict, global_fill: dict, field_fill: dict,
                  global_gt: dict, field_gt: dict) -> Path:

    models = ACTIVE_MODELS
    colors = {m: _PALETTE[i % len(_PALETTE)] for i, m in enumerate(models)}

    def safe(v):
        try: return round(float(v), 1)
        except: return 0.0

    # Données JS
    import json as _json
    js_models    = _json.dumps(models)
    js_colors    = _json.dumps(colors)
    js_imp_vars  = _json.dumps(IMPORTANT_VARS)
    js_latencies = _json.dumps({m: safe(latencies.get(m, 0)) for m in models})
    js_fill      = _json.dumps({m: safe(global_fill.get(m, 0)) for m in models})
    js_gt        = _json.dumps({m: safe(global_gt.get(m, 0)) for m in models})
    js_fill_f    = _json.dumps({
        m: {f: safe(field_fill.get(m, {}).get(f, 0)) for f in IMPORTANT_VARS}
        for m in models
    })
    js_gt_f = _json.dumps({
        m: {f: safe(field_gt.get(m, {}).get(f) or 0) for f in IMPORTANT_VARS}
        for m in models
    })

    html = f"""<!DOCTYPE html>
<html lang="fr">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Benchmark OCR — Cartes grises ES</title>
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.js"></script>
<style>
*{{box-sizing:border-box;margin:0;padding:0}}
body{{font-family:system-ui,sans-serif;background:#f5f7fa;color:#333;font-size:14px}}
header{{background:#1a2744;color:#fff;padding:20px 32px}}
header h1{{font-size:1.25em;font-weight:500;margin-bottom:3px}}
header p{{opacity:.6;font-size:.82em}}
.container{{max-width:1100px;margin:24px auto;padding:0 20px}}
.chips{{display:flex;flex-wrap:wrap;gap:8px;margin-bottom:20px}}
.chip{{padding:5px 14px;border-radius:20px;color:#fff;font-size:.82em;font-weight:500}}
.tabs{{display:flex;gap:4px;flex-wrap:wrap;margin-bottom:16px}}
.tab{{font-size:12px;padding:5px 14px;border-radius:6px;border:0.5px solid #ccc;
      background:transparent;color:#666;cursor:pointer}}
.tab.active{{background:#1a2744;color:#fff;border-color:#1a2744;font-weight:500}}
.sep{{width:1px;background:#ddd;margin:0 4px;align-self:stretch}}
.legend{{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:12px}}
.legend-item{{display:flex;align-items:center;gap:5px;font-size:12px;color:#666}}
.legend-dot{{width:10px;height:10px;border-radius:2px}}
.chart-card{{background:#fff;border:0.5px solid #e0e0e0;border-radius:10px;padding:16px;margin-bottom:20px}}
.note{{font-size:11px;color:#999;margin-top:16px;line-height:1.6}}
</style>
</head>
<body>
<header>
  <h1>Benchmark OCR — Cartes grises espagnoles</h1>
  <p>Généré le {datetime.now().strftime("%d/%m/%Y %H:%M")} · {len(models)} modèles · {len(ALL_VARS)} champs</p>
</header>
<div class="container">
  <div class="chips" id="chips"></div>
  <div class="tabs" id="tabBar"></div>
  <div class="legend" id="legend"></div>
  <div class="chart-card"><div id="chartWrap" style="position:relative"></div></div>
  <p class="note">
    Fill rate = % de champs non vides extraits sur l'ensemble des docs.<br>
    GT accuracy = % de champs exactement corrects vs ground truth (N/A si aucun doc annoté).<br>
    Latence : plus bas = mieux. Fill rate et GT accuracy : plus haut = mieux.
  </p>
</div>
<script>
const MODELS       = {js_models};
const COLORS       = {js_colors};
const IMP_VARS     = {js_imp_vars};
const LATENCIES    = {js_latencies};
const FILL         = {js_fill};
const GT           = {js_gt};
const FILL_FIELD   = {js_fill_f};
const GT_FIELD     = {js_gt_f};

// Onglets globaux
const GLOBAL_TABS = [
  {{ key:'latency',  label:'Latence',          data:LATENCIES, unit:'s',   axis:'x', lowerBetter:true  }},
  {{ key:'fill',     label:'Fill rate global', data:FILL,      unit:'%',   axis:'x', lowerBetter:false }},
  {{ key:'gt',       label:'GT accuracy',      data:GT,        unit:'%',   axis:'x', lowerBetter:false }},
];
// Onglets par champ important
const FIELD_TABS = [
  {{ key:'fill_field', label:'Fill rate par champ',   data:FILL_FIELD, unit:'%' }},
  {{ key:'gt_field',   label:'GT accuracy par champ', data:GT_FIELD,   unit:'%' }},
];

let active = 'fill';
let chart  = null;

// Chips modèles
document.getElementById('chips').innerHTML = MODELS.map(m =>
  `<span class="chip" style="background:${{COLORS[m].bar}}">${{m}}</span>`).join('');

// Légende
document.getElementById('legend').innerHTML = MODELS.map(m =>
  `<span class="legend-item">
    <span class="legend-dot" style="background:${{COLORS[m].bar}}"></span>${{m}}
  </span>`).join('');

// Onglets
const bar = document.getElementById('tabBar');
GLOBAL_TABS.forEach(t => {{
  const btn = document.createElement('button');
  btn.className = 'tab' + (t.key === active ? ' active' : '');
  btn.textContent = t.label; btn.dataset.key = t.key;
  btn.onclick = () => {{ active = t.key; render(); updateTabs(); }};
  bar.appendChild(btn);
}});
const sep = document.createElement('span');
sep.className = 'sep'; bar.appendChild(sep);
FIELD_TABS.forEach(t => {{
  const btn = document.createElement('button');
  btn.className = 'tab'; btn.textContent = t.label; btn.dataset.key = t.key;
  btn.onclick = () => {{ active = t.key; render(); updateTabs(); }};
  bar.appendChild(btn);
}});

function updateTabs() {{
  document.querySelectorAll('.tab').forEach(t =>
    t.classList.toggle('active', t.dataset.key === active));
}}

function render() {{
  const wrap = document.getElementById('chartWrap');
  if (chart) {{ chart.destroy(); chart = null; }}

  // ── Onglet par champ ──────────────────────────────────────────────────────
  const ft = FIELD_TABS.find(t => t.key === active);
  if (ft) {{
    const h = Math.max(IMP_VARS.length * (MODELS.length * 22 + 16) + 80, 300);
    wrap.style.height = h + 'px';
    wrap.innerHTML = `<canvas id="mc" role="img" aria-label="${{ft.label}} par champ" style="display:block"></canvas>`;
    chart = new Chart(document.getElementById('mc'), {{
      type: 'bar',
      data: {{
        labels: IMP_VARS,
        datasets: MODELS.map(m => ({{
          label: m,
          data: IMP_VARS.map(f => ft.data[m]?.[f] ?? 0),
          backgroundColor: COLORS[m].bar + 'CC',
          borderColor: COLORS[m].border,
          borderWidth: 1, borderRadius: 3,
        }}))
      }},
      options: {{
        indexAxis: 'y', responsive: true, maintainAspectRatio: false,
        plugins: {{
          legend: {{ display: true, position: 'top',
            labels: {{ font: {{ size: 11 }}, boxWidth: 12, padding: 10 }} }},
          tooltip: {{ callbacks: {{ label: ctx =>
            ` ${{ctx.dataset.label}}: ${{ctx.parsed.x.toFixed(1)}}${{ft.unit}}` }} }}
        }},
        scales: {{
          x: {{ beginAtZero:true, max:100,
            grid: {{ color:'rgba(0,0,0,0.05)' }},
            ticks: {{ font:{{size:11}}, color:'#888', callback: v => v+ft.unit }} }},
          y: {{ ticks: {{ font:{{size:12}}, color:'#555' }}, grid:{{display:false}} }}
        }}
      }}
    }});
    return;
  }}

  // ── Onglet global ─────────────────────────────────────────────────────────
  const gt = GLOBAL_TABS.find(t => t.key === active);
  const h  = Math.max(MODELS.length * 52 + 60, 160);
  wrap.style.height = h + 'px';
  wrap.innerHTML = `<canvas id="mc" role="img" aria-label="${{gt.label}}" style="display:block"></canvas>`;
  chart = new Chart(document.getElementById('mc'), {{
    type: 'bar',
    data: {{
      labels: MODELS,
      datasets: [{{
        data: MODELS.map(m => gt.data[m] ?? 0),
        backgroundColor: MODELS.map(m => COLORS[m].bar + 'CC'),
        borderColor:     MODELS.map(m => COLORS[m].border),
        borderWidth: 1, borderRadius: 4,
      }}]
    }},
    options: {{
      indexAxis: 'y', responsive: true, maintainAspectRatio: false,
      plugins: {{
        legend: {{ display: false }},
        tooltip: {{ callbacks: {{ label: ctx =>
          ` ${{ctx.parsed.x.toFixed(gt.key === 'latency' ? 2 : 1)}}${{gt.unit}}` }} }}
      }},
      scales: {{
        x: {{ beginAtZero:true,
          grid: {{ color:'rgba(0,0,0,0.05)' }},
          ticks: {{ font:{{size:11}}, color:'#888', callback: v => v+gt.unit }} }},
        y: {{ ticks: {{ font:{{size:12}}, color:'#555' }}, grid:{{display:false}} }}
      }}
    }}
  }});
}}

render();
</script>
</body>
</html>"""

    html_path = RESULTS_REPORT / f"report_{ts}.html"
    RESULTS_REPORT.mkdir(parents=True, exist_ok=True)
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html)
    return html_path


# ─── MAIN ─────────────────────────────────────────────────────────────────────

def run():
    print("📊 Génération du rapport final...\n")

    ts = datetime.now().strftime("%Y%m%d_%H%M%S")

    latencies             = load_latencies()
    global_fill, field_fill = load_fill_rates()
    global_gt,   field_gt   = load_gt_scores()

    # CSV
    csv_path  = write_csv(ts, latencies, global_fill, field_fill, global_gt, field_gt)

    # HTML
    html_path = generate_html(ts, latencies, global_fill, field_fill, global_gt, field_gt)

    # Résumé terminal
    print(f"  {'MODÈLE':<15} {'LATENCE':>8} {'FILL':>8} {'GT ACC':>8}")
    print("  " + "─" * 45)
    for model in ACTIVE_MODELS:
        gt_val = global_gt.get(model)
        gt_str = f"{gt_val:.1f}%" if gt_val else "N/A"
        print(f"  {model:<15} {latencies.get(model, 0):>7.2f}s "
              f"{global_fill.get(model, 0):>7.1f}% {gt_str:>8}")

    print(f"\n  💾  CSV     → {csv_path}")
    print(f"  🌐  HTML    → {html_path}\n")


if __name__ == "__main__":
    run()